# Discrete Flow Matching for 7 × 7 Queens

Transform a fully colored puzzle into a sparse board: **queens retain their original region colors; all other cells become EMPTY**.

Exactly one queen belongs to each row, column and region; queens cannot touch diagonally. Longer diagonals are allowed.

1. Prepared dataset and forward-path visualization.
2. Online categorical flow-matching training.
3. Model saving and loading.
4. Standard stochastic DFM solution trajectories.
5. Strict solution success rate on held-out puzzles.

The dataset is stored in <code>data/queens7/</code>. Running this notebook never generates or searches for new puzzles.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))
import html
from collections import Counter, defaultdict
import json
import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import HTML, display
from plotly.subplots import make_subplots
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, RandomSampler
from _shared.model import Denoiser
from _shared.queens_data import (
    BOARD_SIZE,
    EMPTY,
    load_queens_dataset,
    solution_checks,
    split_puzzles,
)

SEED = 17
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = BOARD_SIZE + 1
FLOW_STEPS = 100
ARTIFACTS = PROJECT_ROOT / "artifacts" / "queens7"
ARTIFACTS.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = ARTIFACTS / "queens7_discrete_flow_matching.pt"
PALETTE = [
    "#B8A5E8",
    "#F0B48B",
    "#92C9E8",
    "#A9D4A4",
    "#E8CD83",
    "#ECA6BB",
    "#9FB9D4",
    "#F1F3F5",
]
CATEGORICAL_SCALE = [
    [position, color]
    for index, color in enumerate(PALETTE)
    for position in (index / NUM_CLASSES, (index + 1) / NUM_CLASSES)
]
PLOT_CONFIG = {"responsive": True, "displaylogo": False, "displayModeBar": False}
display(
    HTML(
        f"<b>Queens 7×7 · Discrete Flow Matching</b><br>"
        f"Device: {html.escape(str(DEVICE))} · EMPTY token: {EMPTY} · {FLOW_STEPS} sampling steps"
    )
)

## 1. Prepared dataset

The collection combines already solved **AMAZE**, **Queens.game** and **Queendom** examples with synthetic pairs prepared offline. Every record includes its original source and queen coordinates.

- External answers are copied from their sources; their uniqueness claim is trusted.
- Synthetic boards were checked for uniqueness once during offline preparation. Synthetic regions are grown round robin, then their boundaries are refined to obtain a unique solution while preserving connectivity and 2?12 cells per region.
- Rotated, reflected and color-renamed duplicates are removed before splitting.
- Only training uses geometric augmentations and color permutations.
- We make our own split; this is not an official AMAZE benchmark.

Sources: [AMAZE data](https://huggingface.co/datasets/piekenius123/Amaze), [AMAZE construction](https://github.com/spatigen/amaze/tree/main/queen-generator), [Queens.game](https://queens.game/about.html), [Queendom](https://github.com/bonesmoses/queendom).

To rebuild outside the notebook, run <code>scripts/collect_queens7.py</code>, then <code>scripts/generate_queens7.py --count 5000</code>.


In [2]:
puzzles = load_queens_dataset(PROJECT_ROOT / "data" / "queens7")
splits = split_puzzles(puzzles["fingerprints"], seed=SEED)
regions, queen_masks = puzzles["regions"], puzzles["queens"]
target_tokens = np.where(queen_masks, regions, EMPTY).astype(np.int64)


class QueensDataset(Dataset):
    def __init__(self, indices, augment=False):
        self.indices = np.asarray(indices, dtype=np.int64)
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, index):
        source_index = int(self.indices[index])
        board = torch.from_numpy(regions[source_index].astype(np.int64))
        queens = torch.from_numpy(queen_masks[source_index].copy())
        if self.augment:
            turns = int(torch.randint(4, ()).item())
            board = torch.rot90(board, turns, dims=(-2, -1))
            queens = torch.rot90(queens, turns, dims=(-2, -1))
            if bool(torch.randint(2, ()).item()):
                board, queens = board.flip(-1), queens.flip(-1)
            board = torch.randperm(BOARD_SIZE)[board]
        return board.contiguous(), torch.where(queens, board, EMPTY).contiguous()


train_dataset = QueensDataset(splits["train"], augment=True)
validation_dataset = QueensDataset(splits["validation"])
test_dataset = QueensDataset(splits["test"])
source_summary = " · ".join(
    f"{html.escape(name)}: {count:,}"
    for name, count in puzzles["metadata"]["source_counts"].items()
)
split_summary = " · ".join(
    f"{name}: {len(indices):,}" for name, indices in splits.items()
)
display(
    HTML(
        f"<b>{len(regions):,} distinct puzzles</b><br>{source_summary}<br>{split_summary}"
    )
)

In [3]:
def add_board(fig, tokens, board, row, col, queens=False):
    tokens, board = np.asarray(tokens), np.asarray(board)
    fig.add_trace(
        go.Heatmap(
            z=tokens,
            zmin=-0.5,
            zmax=NUM_CLASSES - 0.5,
            colorscale=CATEGORICAL_SCALE,
            showscale=False,
            xgap=1,
            ygap=1,
            customdata=board,
            hovertemplate="row=%{y}, col=%{x}<br>region=%{customdata}<br>token=%{z}<extra></extra>",
        ),
        row=row,
        col=col,
    )
    xs, ys = [], []
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE):
            if c + 1 < BOARD_SIZE and board[r, c] != board[r, c + 1]:
                xs.extend([c + 0.5, c + 0.5, None])
                ys.extend([r - 0.5, r + 0.5, None])
            if r + 1 < BOARD_SIZE and board[r, c] != board[r + 1, c]:
                xs.extend([c - 0.5, c + 0.5, None])
                ys.extend([r + 0.5, r + 0.5, None])
    fig.add_trace(
        go.Scatter(
            x=xs,
            y=ys,
            mode="lines",
            line=dict(color="#64748B", width=1.5),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    if queens:
        qr, qc = np.where(tokens != EMPTY)
        fig.add_trace(
            go.Scatter(
                x=qc,
                y=qr,
                mode="text",
                text=["♛"] * len(qr),
                textfont=dict(size=25, color="#172033"),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row,
            col=col,
        )
    fig.update_xaxes(
        range=[-0.5, BOARD_SIZE - 0.5],
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        constrain="domain",
        fixedrange=True,
        row=row,
        col=col,
    )
    xaxis_name = fig.get_subplot(row, col).xaxis.plotly_name.replace("axis", "")
    fig.update_yaxes(
        range=[BOARD_SIZE - 0.5, -0.5],
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        scaleanchor=xaxis_name,
        scaleratio=1,
        constrain="domain",
        fixedrange=True,
        row=row,
        col=col,
    )


def style_boards(fig, title, rows=1):
    fig.update_layout(
        title=dict(text=title, x=0, font=dict(size=22)),
        template="plotly_white",
        autosize=True,
        height=rows * 225 + 90,
        margin=dict(l=10, r=10, t=85, b=20),
        font=dict(family="Arial, sans-serif", size=15),
        showlegend=False,
    )
    fig.update_annotations(font=dict(size=14))
    return fig

### 1.1 Examples from every source

Draw up to two distinct examples per source before augmentation, with the supplied queens directly on the complete region coloring. Queendom contributes one example.


In [4]:
SOURCE_EXAMPLES = 2
source_example_rng = np.random.default_rng(SEED + 201)
dataset_source_examples = {}
for source_name in sorted(set(puzzles["source_names"].tolist())):
    available = np.flatnonzero(puzzles["source_names"] == source_name)
    chosen = source_example_rng.choice(
        available,
        size=min(SOURCE_EXAMPLES, len(available)),
        replace=False,
    )
    dataset_source_examples[source_name] = chosen
    fig = make_subplots(
        rows=1,
        cols=len(chosen),
        subplot_titles=[f"Example {number}" for number in range(1, len(chosen) + 1)],
        horizontal_spacing=0.07,
    )
    for col, index in enumerate(chosen, start=1):
        add_board(fig, regions[index], regions[index], 1, col)
        queen_rows, queen_cols = np.where(queen_masks[index])
        fig.add_trace(
            go.Scatter(
                x=queen_cols,
                y=queen_rows,
                mode="text",
                text=[chr(0x265B)] * len(queen_rows),
                textfont=dict(size=25, color="#172033"),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=1,
            col=col,
        )
    style_boards(
        fig,
        f"{source_name} ? {len(available):,} puzzles",
        rows=1,
    ).show(config=PLOT_CONFIG)

### 1.2 Number of colorings per solution

Group the entire prepared dataset by the exact queen positions: the queen's column in each row.
Let $K(s)$ be the number of distinct region maps assigned to solution $s$. Under a uniform choice
among the distinct observed solutions,

$$
P(K=k)=\frac{\#\{s:K(s)=k\}}{\#\{\text{distinct observed solutions}\}}.
$$

This is the empirical distribution in our dataset, with no augmentation. The dataset has already
removed duplicate region maps up to rotations, reflections and color relabeling.
Here queen positions retain their exact orientation; rotated queen placements count as different solutions.


In [5]:
colorings_by_solution = defaultdict(list)
for index, record in enumerate(puzzles["records"]):
    queen_columns = tuple(column for row, column in sorted(record["queens"]))
    colorings_by_solution[queen_columns].append(index)

coloring_multiplicities = np.asarray(
    [len(indices) for indices in colorings_by_solution.values()],
    dtype=np.int64,
)
multiplicity_counts = Counter(coloring_multiplicities.tolist())
multiplicity_support = np.arange(1, int(coloring_multiplicities.max()) + 1)
multiplicity_frequencies = np.asarray(
    [multiplicity_counts.get(int(count), 0) for count in multiplicity_support]
)
multiplicity_probabilities = multiplicity_frequencies / len(colorings_by_solution)
mean_colorings = float(coloring_multiplicities.mean())
median_colorings = float(np.median(coloring_multiplicities))

fig = go.Figure(
    go.Bar(
        x=multiplicity_support,
        y=multiplicity_probabilities,
        customdata=multiplicity_frequencies,
        marker_color="#5267DB",
        hovertemplate=(
            "Colorings per solution: %{x}<br>"
            "Distinct solutions: %{customdata}<br>"
            "Probability: %{y:.2%}<extra></extra>"
        ),
    )
)
fig.add_vline(
    x=mean_colorings,
    line_color="#DC6557",
    line_width=2,
    line_dash="dash",
    annotation_text=f"Mean = {mean_colorings:.2f}",
    annotation_position="top right",
    annotation_font_color="#B85045",
)
fig.update_layout(
    title=dict(
        text=f"Colorings per solution · {len(regions):,} puzzles · "
        f"{len(colorings_by_solution):,} distinct queen placements",
        x=0,
    ),
    template="plotly_white",
    autosize=True,
    height=440,
    xaxis_title="Number of distinct colorings per solution",
    yaxis_title="Probability · uniformly selected distinct solution",
    font=dict(family="Arial, sans-serif", size=15),
    margin=dict(l=70, r=25, t=85, b=65),
    bargap=0.18,
)
fig.update_xaxes(
    dtick=max(1, int(np.ceil(multiplicity_support[-1] / 30))),
    range=[0.3, float(multiplicity_support[-1]) + 0.7],
)
fig.update_yaxes(tickformat=".0%", rangemode="tozero")
fig.show(config=PLOT_CONFIG)
display(
    HTML(
        f"<b>{len(colorings_by_solution):,} unique queen placements</b> · "
        f"mean {mean_colorings:.2f} colorings · median {median_colorings:g} · "
        f"range {coloring_multiplicities.min()}–{coloring_multiplicities.max()}<br>"
        f"More than one coloring: {np.mean(coloring_multiplicities > 1):.2%} of solutions."
    )
)

## 2. Conditional forward path

For original coloring $c$, the source is $x_0=c$. The supplied solution defines

$$
x_1^i =
\begin{cases}
c^i, & \text{if cell } i \text{ contains a queen},\\
\mathrm{EMPTY}, & \text{otherwise}.
\end{cases}
$$

The linear categorical path is

$$
p_t(x_t^i\mid c,x_1)=(1-t)\delta_{c^i}(x_t^i)+t\delta_{x_1^i}(x_t^i).
$$

Each cell independently chooses its target token with probability $t$. Queens keep their color; other cells switch to EMPTY. A single uniform switch time per cell gives a consistent visualized trajectory.

The original coloring $c$ is always available as a condition. The supplied queen mask is never an input condition.


In [6]:
def sample_conditional_path(source, target, times, generator=None):
    switches = torch.rand(source.shape, device=source.device, generator=generator)
    return torch.where(switches < times[:, None, None], target, source)


FORWARD_TIMES = np.linspace(0, 1, 6)
forward_index = int(splits["validation"][0])
forward_source, forward_target = regions[forward_index], target_tokens[forward_index]
forward_switches = np.random.default_rng(SEED + 101).random((BOARD_SIZE, BOARD_SIZE))
fig = make_subplots(
    rows=1,
    cols=len(FORWARD_TIMES),
    subplot_titles=[f"t = {time:.1f}" for time in FORWARD_TIMES],
    horizontal_spacing=0.025,
)
for col, time in enumerate(FORWARD_TIMES, start=1):
    frame = np.where(forward_switches < time, forward_target, forward_source)
    add_board(fig, frame, forward_source, 1, col, queens=time == 1)
style_boards(fig, "Forward path · fully colored puzzle → supplied solution").show(
    config=PLOT_CONFIG
)

## 3. Conditional endpoint model

Reuse the shared U-Net at resolution $7 \to 4 \to 2 \to 4 \to 7$.

- Current state: **8 one-hot channels**, including EMPTY.
- Original region coloring: **7 one-hot channels**.
- Row and column coordinates: **2 channels**.
- Input: $(B,17,7,7)$; output: $(B,8,7,7)$ endpoint logits.
- Continuous time is embedded sinusoidally and added inside each residual block.

A colored cell can end as its own color or EMPTY; an already empty cell must stay EMPTY under this path. Logits enforce this support. Row, column, region and queen-count constraints are learned, not imposed.


In [7]:
MODEL_CONFIG = dict(
    board_size=BOARD_SIZE,
    base_channels=64,
    time_dim=128,
    time_embedding_scale=100.0,
)


class QueensDFM(nn.Module):
    def __init__(
        self, board_size=7, base_channels=64, time_dim=128, time_embedding_scale=100.0
    ):
        super().__init__()
        if board_size != BOARD_SIZE:
            raise ValueError("This experiment and dataset use 7×7 boards.")
        self.time_embedding_scale = float(time_embedding_scale)
        self.network = Denoiser(
            embedding_dim=NUM_CLASSES + BOARD_SIZE + 2,
            output_dim=NUM_CLASSES,
            base_channels=base_channels,
            time_dim=time_dim,
        )
        positions = torch.linspace(-1, 1, board_size)
        rr, cc = torch.meshgrid(positions, positions, indexing="ij")
        self.register_buffer("coordinates", torch.stack((rr, cc))[None])

    def forward(self, state, coloring, times):
        state_channels = (
            F.one_hot(state.long(), NUM_CLASSES).permute(0, 3, 1, 2).float()
        )
        color_channels = (
            F.one_hot(coloring.long(), BOARD_SIZE).permute(0, 3, 1, 2).float()
        )
        inputs = torch.cat(
            (
                state_channels,
                color_channels,
                self.coordinates.expand(len(state), -1, -1, -1),
            ),
            dim=1,
        )
        logits = self.network(inputs, times * self.time_embedding_scale)
        allowed = F.one_hot(coloring.long(), NUM_CLASSES).permute(0, 3, 1, 2).bool()
        allowed = allowed & (state != EMPTY)[:, None]
        allowed[:, EMPTY] = True
        return logits.masked_fill(~allowed, -torch.inf)


model = QueensDFM(**MODEL_CONFIG).to(DEVICE)
display(
    HTML(
        f"<b>Conditional U-Net</b> · {sum(p.numel() for p in model.parameters()):,} parameters"
        " · (B, 17, 7, 7) → (B, 8, 7, 7)"
    )
)

## 4. Online training

Independently for every sample, draw an augmented puzzle/solution pair, $t\sim U[0,1)$ and per-cell path switches. Predict the endpoint from $(x_t,c,t)$ with mean pixel cross-entropy:

$$
\mathcal L=-\frac{1}{B\cdot49}
\sum_{b=1}^{B}\sum_{i=1}^{49}
\log p_\theta(x_{1,b}^i\mid x_{t,b},c_b,t_b).
$$

Known EMPTY cells contribute zero loss. The loss plot updates in place on a logarithmic scale. Validation uses fixed times and random switches; a separate $t=0$ curve evaluates the untouched input puzzle.

Run the setup cell before training or loading. To resume, load a checkpoint and then run the training loop.


In [8]:
BATCH_SIZE = 128
EPOCHS = 440  # Additional epochs for each execution of the training loop.
SAMPLES_PER_EPOCH = 4096  # Random augmented views, sampled with replacement.
LEARNING_RATE = 2e-4
PLOT_EVERY = 10

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=RandomSampler(
        train_dataset, replacement=True, num_samples=SAMPLES_PER_EPOCH
    ),
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
history = dict(
    step=[],
    train_ce=[],
    validation_step=[],
    validation_ce=[],
    validation_t0_ce=[],
    epochs_completed=0,
)
global_step = 0


def training_loss(coloring, target):
    times = torch.rand(len(coloring), device=DEVICE)
    state = sample_conditional_path(coloring, target, times)
    return F.cross_entropy(model(state, coloring, times), target)


@torch.no_grad()
def validation_losses():
    was_training = model.training
    model.eval()
    generator = torch.Generator(device=DEVICE).manual_seed(SEED + 401)
    total, count, at_zero, count_at_zero = 0.0, 0, 0.0, 0
    try:
        for coloring, target in validation_loader:
            coloring, target = coloring.to(DEVICE), target.to(DEVICE)
            for time in (0.0, 0.2, 0.4, 0.6, 0.8, 0.95):
                times = torch.full((len(coloring),), time, device=DEVICE)
                state = sample_conditional_path(coloring, target, times, generator)
                loss = F.cross_entropy(model(state, coloring, times), target).item()
                total += loss * len(coloring)
                count += len(coloring)
                if time == 0.0:
                    at_zero += loss * len(coloring)
                    count_at_zero += len(coloring)
        return total / count, at_zero / count_at_zero
    finally:
        model.train(was_training)

In [9]:
training_progress = widgets.IntProgress(
    value=0,
    min=0,
    max=EPOCHS * len(train_loader),
    description="Training",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
training_status = widgets.HTML()
loss_plot = go.FigureWidget()
loss_plot.add_scatter(
    name="Train CE · moving mean", mode="lines", line=dict(color="#4F6BED", width=2)
)
loss_plot.add_scatter(
    name="Validation CE · fixed path",
    mode="lines+markers",
    line=dict(color="#E49B43", width=2),
    marker=dict(size=5),
)
loss_plot.add_scatter(
    name="Validation CE at t = 0",
    mode="lines+markers",
    line=dict(color="#2F9D88", width=2, dash="dot"),
    marker=dict(size=5),
)
loss_plot.update_layout(
    title="Queens DFM · endpoint cross-entropy",
    template="plotly_white",
    xaxis_title="Optimizer step",
    yaxis_title="Mean pixel CE · log scale",
    yaxis_type="log",
    autosize=True,
    height=450,
    margin=dict(l=65, r=20, t=65, b=90),
    legend=dict(orientation="h", y=-0.2, x=0),
    font=dict(family="Arial, sans-serif", size=15),
)
display(training_progress, training_status, loss_plot)


def refresh_loss_plot():
    values = np.asarray(history["train_ce"], dtype=float)
    window = min(25, len(values))
    if not window:
        return
    smooth = np.convolve(values, np.ones(window) / window, mode="valid")
    with loss_plot.batch_update():
        loss_plot.data[0].x = history["step"][window - 1 :]
        loss_plot.data[0].y = np.maximum(smooth, 1e-12)
        loss_plot.data[1].x = history["validation_step"]
        loss_plot.data[1].y = np.maximum(history["validation_ce"], 1e-12)
        loss_plot.data[2].x = history["validation_step"]
        loss_plot.data[2].y = np.maximum(history["validation_t0_ce"], 1e-12)


refresh_loss_plot()
start_epoch = int(history["epochs_completed"])
for local_epoch in range(EPOCHS):
    model.train()
    for coloring, target in train_loader:
        coloring, target = coloring.to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = training_loss(coloring, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        global_step += 1
        history["step"].append(global_step)
        history["train_ce"].append(float(loss.item()))
        training_progress.value += 1
        if global_step % PLOT_EVERY == 0 or len(history["step"]) == 1:
            refresh_loss_plot()
            training_status.value = (
                f"Epoch {start_epoch + local_epoch + 1} · step {global_step:,} · "
                f"train CE {history['train_ce'][-1]:.5f}"
            )
    val_ce, val_t0_ce = validation_losses()
    history["validation_step"].append(global_step)
    history["validation_ce"].append(val_ce)
    history["validation_t0_ce"].append(val_t0_ce)
    history["epochs_completed"] = start_epoch + local_epoch + 1
    refresh_loss_plot()
    training_status.value = (
        f"Epoch {history['epochs_completed']} · step {global_step:,} · "
        f"validation CE {val_ce:.5f} · t = 0 CE {val_t0_ce:.5f}"
    )
model.eval()
training_progress.bar_style = "success"

IntProgress(value=0, description='Training', layout=Layout(width='100%'), max=14080, style=ProgressStyle(descr…

HTML(value='')

FigureWidget({
    'data': [{'line': {'color': '#4F6BED', 'width': 2},
              'mode': 'lines',
              'name': 'Train CE · moving mean',
              'type': 'scatter',
              'uid': '327ecb1a-3d08-41e5-9973-4342a89f6804'},
             {'line': {'color': '#E49B43', 'width': 2},
              'marker': {'size': 5},
              'mode': 'lines+markers',
              'name': 'Validation CE · fixed path',
              'type': 'scatter',
              'uid': '28d146c8-e63b-480e-ba5d-97b57db5a37b'},
             {'line': {'color': '#2F9D88', 'dash': 'dot', 'width': 2},
              'marker': {'size': 5},
              'mode': 'lines+markers',
              'name': 'Validation CE at t = 0',
              'type': 'scatter',
              'uid': 'f37ba906-1b7d-46ae-8e6a-9f08fe740c4d'}],
    'layout': {'autosize': True,
               'font': {'family': 'Arial, sans-serif', 'size': 15},
               'height': 450,
               'legend': {'orientation': 'h', 'x': 0, 

## 5. Save and load

### Save

Save weights, optimizer, loss history, exact dataset fingerprint and train/validation/test split to the root <code>artifacts/queens7/</code> directory.


In [10]:
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "format_version": "queens7_dfm_v1",
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "model_config": MODEL_CONFIG,
        "history": history,
        "global_step": global_step,
        "flow_steps": FLOW_STEPS,
        "seed": SEED,
        "dataset_sha256": puzzles["metadata"]["data_sha256"],
        "splits": {
            name: torch.as_tensor(indices.copy()) for name, indices in splits.items()
        },
        "torch_rng_state": torch.get_rng_state(),
        "cuda_rng_state_all": (
            torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else []
        ),
    },
    CHECKPOINT_PATH,
)
display(
    HTML(
        f"Saved: <code>{html.escape(str(CHECKPOINT_PATH.relative_to(PROJECT_ROOT)))}</code>"
    )
)

### Load

Run Sections 1–3 and the training setup cell, then skip the training loop and save cell. Load here to continue directly to sampling, or run the training loop to resume. Restore the saved split and reject mismatched dataset files.


In [11]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
if checkpoint["format_version"] != "queens7_dfm_v1":
    raise ValueError("Unsupported Queens checkpoint format.")
if checkpoint["dataset_sha256"] != puzzles["metadata"]["data_sha256"]:
    raise ValueError("Checkpoint and dataset differ; restore the matching dataset.")
MODEL_CONFIG = checkpoint["model_config"]
model = QueensDFM(**MODEL_CONFIG).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
history, global_step = checkpoint["history"], int(checkpoint["global_step"])
FLOW_STEPS, SEED = int(checkpoint["flow_steps"]), int(checkpoint["seed"])
splits = {name: value.numpy().copy() for name, value in checkpoint["splits"].items()}
train_dataset = QueensDataset(splits["train"], augment=True)
validation_dataset = QueensDataset(splits["validation"])
test_dataset = QueensDataset(splits["test"])
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=RandomSampler(
        train_dataset, replacement=True, num_samples=SAMPLES_PER_EPOCH
    ),
    num_workers=0,
    pin_memory=DEVICE.type == "cuda",
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)
torch.set_rng_state(checkpoint["torch_rng_state"])
if DEVICE.type == "cuda" and checkpoint["cuda_rng_state_all"]:
    if len(checkpoint["cuda_rng_state_all"]) == torch.cuda.device_count():
        torch.cuda.set_rng_state_all(checkpoint["cuda_rng_state_all"])
model.eval()
display(
    HTML(
        f"<b>Loaded Queens DFM</b> · {history['epochs_completed']} epochs · "
        f"{global_step:,} optimizer steps"
    )
)

## 6. Stochastic solution trajectories

For the linear categorical path, a still-colored cell jumps to EMPTY with probability

$$
P(x_{t+\Delta t}^i=\mathrm{EMPTY}\mid x_t,c)
=\frac{\Delta t}{1-t}\,
p_\theta(x_1^i=\mathrm{EMPTY}\mid x_t,c,t).
$$

An empty cell stays empty. Every step evaluates the network once and independently samples the categorical jumps. The same puzzle always starts from its full coloring; different random seeds produce different trajectories.

This is the finite-step Euler approximation to the DFM jump process. A mistaken erasure cannot be undone. Outputs receive no backtracking, rule-based repair, top-k selection or forced count of seven queens.

Reference: [Discrete Flow Matching, §2.3–2.4](https://arxiv.org/html/2407.15595v2).


In [12]:
@torch.no_grad()
def sample_queens(
    candidate, coloring, steps=100, seed=0, capture_steps=(), progress=None
):
    if not isinstance(steps, int) or steps < 1:
        raise ValueError("steps must be a positive integer.")
    device = next(candidate.parameters()).device
    coloring = torch.as_tensor(coloring, dtype=torch.long, device=device)
    if coloring.ndim == 2:
        coloring = coloring[None]
    if coloring.ndim != 3 or tuple(coloring.shape[-2:]) != (BOARD_SIZE, BOARD_SIZE):
        raise ValueError("Expected coloring with shape (B,7,7) or (7,7).")
    if bool(((coloring < 0) | (coloring >= BOARD_SIZE)).any()):
        raise ValueError("Coloring must contain only the seven original region IDs.")
    state = coloring.clone()
    generator = torch.Generator(device=device).manual_seed(int(seed))
    capture_steps = set(map(int, capture_steps))
    if any(step < 0 or step > steps for step in capture_steps):
        raise ValueError("Capture steps must be between zero and steps.")
    snapshots = {0: state.cpu().clone()} if 0 in capture_steps else {}
    was_training = candidate.training
    candidate.eval()
    try:
        for step in range(steps):
            time = step / steps
            times = torch.full((len(state),), time, device=device)
            p_empty = candidate(state, coloring, times).softmax(dim=1)[:, EMPTY]
            # dt/(1-t) = 1/(steps-step); the final factor is exactly one.
            jump_probability = p_empty / (steps - step)
            jump = (
                torch.rand(state.shape, device=device, generator=generator)
                < jump_probability
            )
            state = torch.where(jump & (state != EMPTY), EMPTY, state)
            if step + 1 in capture_steps:
                snapshots[step + 1] = state.cpu().clone()
            if progress is not None:
                progress(step + 1, steps)
    finally:
        candidate.train(was_training)
    return state.cpu(), snapshots

In [13]:
N_TRAJECTORY_PUZZLES = 3
TRAJECTORY_ATTEMPTS = 2
CAPTURE_STEPS = tuple(sorted(set(np.rint(np.linspace(0, FLOW_STEPS, 6)).astype(int))))
trajectory_indices = splits["test"][:N_TRAJECTORY_PUZZLES]
trajectory_progress = widgets.IntProgress(
    value=0,
    max=len(trajectory_indices) * TRAJECTORY_ATTEMPTS * FLOW_STEPS,
    description="Trajectories",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
display(trajectory_progress)
trajectory_records = []
for puzzle_index in trajectory_indices:
    for attempt in range(TRAJECTORY_ATTEMPTS):
        offset = len(trajectory_records) * FLOW_STEPS

        def update_trajectory(done, total, offset=offset):
            trajectory_progress.value = offset + done

        final, snapshots = sample_queens(
            model,
            regions[puzzle_index],
            steps=FLOW_STEPS,
            seed=SEED + 1000 + len(trajectory_records),
            capture_steps=CAPTURE_STEPS,
            progress=update_trajectory,
        )
        solved = bool(
            solution_checks(regions[puzzle_index], final[0].numpy())["solved"]
        )
        trajectory_records.append((puzzle_index, attempt, snapshots, solved))
trajectory_progress.bar_style = "success"

for puzzle_index, attempt, snapshots, solved in trajectory_records:
    titles = [f"t = {step / FLOW_STEPS:.1f}" for step in CAPTURE_STEPS] + [
        "Supplied solution"
    ]
    fig = make_subplots(
        rows=1, cols=len(titles), subplot_titles=titles, horizontal_spacing=0.018
    )
    for col, step in enumerate(CAPTURE_STEPS, start=1):
        add_board(
            fig,
            snapshots[step][0].numpy(),
            regions[puzzle_index],
            1,
            col,
            queens=step == FLOW_STEPS,
        )
    add_board(
        fig,
        target_tokens[puzzle_index],
        regions[puzzle_index],
        1,
        len(titles),
        queens=True,
    )
    source = puzzles["source_names"][puzzle_index]
    status = "SOLVED" if solved else "UNSOLVED"
    style_boards(
        fig, f"{source} · test puzzle {puzzle_index} · attempt {attempt + 1} · {status}"
    ).show(config=PLOT_CONFIG)

IntProgress(value=0, description='Trajectories', layout=Layout(width='100%'), max=600, style=ProgressStyle(des…

## 7. How often is a held-out puzzle solved?

Run independent stochastic trajectories from each selected test puzzle's full coloring.

- **Solve probability:** average success over puzzles and attempts.
- **Solved at least once:** fraction of puzzles solved by any of the repeated attempts.
- **Exact match:** all output tokens equal the supplied target.
- Strict scoring checks every Queens rule on the raw output. Checking a proposed answer is linear-time; it does not search for solutions or verify uniqueness.

The 95% interval bootstraps whole puzzles to preserve the grouping of repeated attempts. Source-specific rates separate externally collected and synthetic puzzles.

The retry curve shows the probability of finding at least one solution within k attempts. It averages over all subsets of k recorded attempts for each puzzle, then across puzzles. The shaded band is a 95% puzzle-bootstrap interval. The curve stops at the number of attempts actually sampled.

The orange dashed reference assumes the same single-attempt probability $\bar p$ for every puzzle and shows $1-(1-\bar p)^k$. The blue curve estimates success separately for each puzzle before averaging.


In [14]:
N_EVAL_PUZZLES = 256  # None evaluates every held-out puzzle.
ATTEMPTS_PER_PUZZLE = 32
EVAL_BATCH_SIZE = 128
EVAL_SEED = SEED + 10_000
if ATTEMPTS_PER_PUZZLE < 1 or EVAL_BATCH_SIZE < 1:
    raise ValueError("Attempt count and batch size must be positive.")
evaluation_rng = np.random.default_rng(EVAL_SEED)
evaluation_indices = evaluation_rng.permutation(splits["test"])
if N_EVAL_PUZZLES is not None:
    if N_EVAL_PUZZLES < 1:
        raise ValueError("N_EVAL_PUZZLES must be positive or None.")
    evaluation_indices = evaluation_indices[:N_EVAL_PUZZLES]
flat_indices = np.repeat(evaluation_indices, ATTEMPTS_PER_PUZZLE)
evaluation_progress = widgets.IntProgress(
    value=0,
    max=len(flat_indices) * FLOW_STEPS,
    description="Evaluation",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
evaluation_status = widgets.HTML()
display(evaluation_progress, evaluation_status)
predicted_batches = []
for start in range(0, len(flat_indices), EVAL_BATCH_SIZE):
    batch_indices = flat_indices[start : start + EVAL_BATCH_SIZE]
    offset = start * FLOW_STEPS

    def update_evaluation(done, total, offset=offset, count=len(batch_indices)):
        evaluation_progress.value = offset + done * count

    prediction, _ = sample_queens(
        model,
        regions[batch_indices],
        steps=FLOW_STEPS,
        seed=EVAL_SEED + start,
        progress=update_evaluation,
    )
    predicted_batches.append(prediction.numpy())
    evaluation_status.value = (
        f"{start + len(batch_indices):,} / {len(flat_indices):,} trajectories"
    )
evaluation_progress.bar_style = "success"

predicted_tokens = np.concatenate(predicted_batches, axis=0)
checks = solution_checks(regions[flat_indices], predicted_tokens)
successes = checks["solved"].reshape(len(evaluation_indices), ATTEMPTS_PER_PUZZLE)
exact_matches = np.all(
    predicted_tokens == target_tokens[flat_indices], axis=(-2, -1)
).reshape(successes.shape)
per_puzzle_success = successes.mean(axis=1)
solve_probability = float(per_puzzle_success.mean())
bootstrap_rng = np.random.default_rng(EVAL_SEED + 1)
bootstrap_means = per_puzzle_success[
    bootstrap_rng.integers(
        0, len(per_puzzle_success), size=(3000, len(per_puzzle_success))
    )
].mean(axis=1)
lower, upper = np.quantile(bootstrap_means, [0.025, 0.975])
at_least_once = float(successes.any(axis=1).mean())
display(
    HTML(
        f"<b>Solved: {100 * solve_probability:.2f}%</b>"
        f" · 95% puzzle-bootstrap interval [{100 * lower:.2f}%, {100 * upper:.2f}%]<br>"
        f"Exact target match: {100 * exact_matches.mean():.2f}% · "
        f"Solved at least once in {ATTEMPTS_PER_PUZZLE} attempts: {100 * at_least_once:.2f}%<br>"
        f"{len(evaluation_indices)} unseen puzzles · {len(flat_indices):,} trajectories · "
        f"{FLOW_STEPS} steps · mean queen count {checks['queen_count'].mean():.2f}"
    )
)
evaluation_results = dict(
    dataset_indices=evaluation_indices,
    predicted_tokens=predicted_tokens.reshape(
        len(evaluation_indices), ATTEMPTS_PER_PUZZLE, BOARD_SIZE, BOARD_SIZE
    ),
    successes=successes,
    exact_matches=exact_matches,
    per_puzzle_success=per_puzzle_success,
    steps=FLOW_STEPS,
)

IntProgress(value=0, description='Evaluation', layout=Layout(width='100%'), max=819200, style=ProgressStyle(de…

HTML(value='')

In [15]:
source_names = puzzles["source_names"][evaluation_indices]
names = ["All test puzzles"] + sorted(set(source_names.tolist()))
means, lows, highs, sample_counts = [], [], [], []
for name in names:
    values = (
        per_puzzle_success
        if name == "All test puzzles"
        else per_puzzle_success[source_names == name]
    )
    draw = bootstrap_rng.integers(0, len(values), size=(3000, len(values)))
    interval = np.quantile(values[draw].mean(axis=1), [0.025, 0.975])
    means.append(100 * float(values.mean()))
    lows.append(100 * float(interval[0]))
    highs.append(100 * float(interval[1]))
    sample_counts.append(len(values))

# For a puzzle with c successes in n recorded attempts, average over all
# subsets of k attempts: pass@k = 1 - C(n-c, k) / C(n, k).
# This uses the actual observed attempts and does not extrapolate beyond n.
attempt_count = successes.shape[1]
attempt_counts = np.arange(attempt_count + 1)
per_puzzle_pass_at_k = np.zeros((len(successes), attempt_count + 1))
no_solution_probability = np.ones(len(successes))
failed_attempts = attempt_count - successes.sum(axis=1)
for k in range(1, attempt_count + 1):
    no_solution_probability *= np.maximum(failed_attempts - k + 1, 0) / (
        attempt_count - k + 1
    )
    per_puzzle_pass_at_k[:, k] = 1.0 - no_solution_probability
pass_at_k = per_puzzle_pass_at_k.mean(axis=0)
mean_single_attempt = float(successes.mean())
mean_probability_pass_at_k = 1.0 - (1.0 - mean_single_attempt) ** attempt_counts

# Resample whole puzzles, keeping their repeated attempts together.
# Bootstrap weights avoid allocating a (bootstrap, puzzles, attempts) array.
retry_rng = np.random.default_rng(EVAL_SEED + 2)
retry_bootstrap_weights = retry_rng.multinomial(
    len(successes),
    np.full(len(successes), 1.0 / len(successes)),
    size=3000,
)
retry_bootstrap = retry_bootstrap_weights @ per_puzzle_pass_at_k / len(successes)
pass_at_k_lower, pass_at_k_upper = np.quantile(
    retry_bootstrap,
    [0.025, 0.975],
    axis=0,
)

fig = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.12,
    subplot_titles=[
        "Solve probability - 95% intervals",
        "At least one solution vs number of attempts",
    ],
)
fig.add_trace(
    go.Bar(
        x=[f"{name}<br>n = {count}" for name, count in zip(names, sample_counts)],
        y=means,
        marker_color=["#4F6BED"] + ["#6CAEA0"] * (len(names) - 1),
        error_y=dict(
            type="data",
            symmetric=False,
            array=np.maximum(0, np.asarray(highs) - means),
            arrayminus=np.maximum(0, np.asarray(means) - lows),
        ),
        hovertemplate="%{x}<br>solved %{y:.2f}%<extra></extra>",
        showlegend=False,
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=attempt_counts,
        y=100 * pass_at_k_lower,
        mode="lines",
        line=dict(width=0),
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=attempt_counts,
        y=100 * pass_at_k_upper,
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor="rgba(79,107,237,0.16)",
        hoverinfo="skip",
        showlegend=False,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=attempt_counts,
        y=100 * pass_at_k,
        mode="lines+markers",
        line=dict(color="#4F6BED", width=3),
        marker=dict(size=7),
        customdata=np.column_stack((100 * pass_at_k_lower, 100 * pass_at_k_upper)),
        hovertemplate=(
            "Attempts: %{x}<br>At least one solution: %{y:.2f}%"
            "<br>95% interval: [%{customdata[0]:.2f}%, %{customdata[1]:.2f}%]"
            "<extra></extra>"
        ),
        name="Per-puzzle estimate",
        showlegend=True,
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=attempt_counts,
        y=100 * mean_probability_pass_at_k,
        mode="lines+markers",
        name="From mean success probability",
        line=dict(color="#D88437", width=3, dash="dash"),
        marker=dict(size=6),
        hovertemplate=(
            "Attempts: %{x}<br>From mean probability: %{y:.2f}%" "<extra></extra>"
        ),
        showlegend=True,
    ),
    row=1,
    col=2,
)
fig.update_yaxes(title_text="Solved trajectories (%)", range=[0, 105], row=1, col=1)
fig.update_yaxes(
    title_text="Probability of at least one solution (%)",
    rangemode="tozero",
    row=1,
    col=2,
)
fig.update_xaxes(
    title_text="Number of attempts",
    dtick=max(1, attempt_count // 10),
    range=[0, attempt_count],
    row=1,
    col=2,
)
fig.update_layout(
    title=dict(text=f"Queens DFM - held-out results - {FLOW_STEPS} steps", x=0),
    template="plotly_white",
    autosize=True,
    height=520,
    legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.28, yanchor="top"),
    bargap=0.2,
    margin=dict(l=65, r=20, t=100, b=140),
    font=dict(family="Arial, sans-serif", size=16),
)
fig.show(config=PLOT_CONFIG)
evaluation_results.update(
    mean_single_attempt=mean_single_attempt,
    mean_probability_pass_at_k=mean_probability_pass_at_k,
    attempt_counts=attempt_counts,
    pass_at_k=pass_at_k,
    pass_at_k_lower=pass_at_k_lower,
    pass_at_k_upper=pass_at_k_upper,
)
np.savez_compressed(
    ARTIFACTS / "queens7_dfm_evaluation.npz",
    **evaluation_results,
    dataset_sha256=puzzles["metadata"]["data_sha256"],
    source_names=source_names,
    evaluation_seed=EVAL_SEED,
)

### 7.1 From retry success to puzzle difficulty

The orange dashed curve uses the mean single-attempt success probability $\bar p$:

$$
P_{\mathrm{mean}}(k)=1-(1-\bar p)^k.
$$

This assumes every puzzle has the same success probability. For puzzles with individual probabilities $p_i$, independent retries on the same puzzle instead give

$$
P_{\mathrm{puzzles}}(k)
=\mathbb E_i\!\left[1-(1-p_i)^k\right].
$$

For $k\ge2$, the function $1-(1-p)^k$ is concave, so

$$
\mathbb E_i\!\left[1-(1-p_i)^k\right]
\le 1-\left(1-\mathbb E_i[p_i]\right)^k.
$$

Some puzzles are already relatively easy and are often solved repeatedly; others remain difficult across retries. Pooling them into one mean success probability overestimates the benefit of repeated attempts when their probabilities differ.

A gap between the orange and blue curves therefore motivates a study of **different puzzle difficulties for this model and sampler**. We estimate those differences from 32 independent attempts per puzzle; finite sampling also contributes uncertainty.


### 7.2 Solvability study

Estimate each held-out puzzle's solvability as its fraction of successful attempts:

$$
\widehat p_i=\frac{\text{successful attempts for puzzle }i}{32}.
$$

Plot a continuous, boundary-corrected kernel density of these estimates. The x-axis is **probability to be solved**; the y-axis is **% of puzzles on a logarithmic scale**, expressed per one percentage point of x so that the area under the density equals 100%.

The smoothing bandwidth defaults to one attempt's resolution ($100/32=3.125$ percentage points). This smooths the measured probabilities; it does not increase the information in 32 attempts. Zero observed successes means no successful attempt was observed for that puzzle. Very small density tails below 0.001% per percentage point are hidden on the log plot.

Rerun the evaluation cell with 32 attempts before this study. The three marked probabilities correspond to the example puzzles below.


In [16]:
study_successes = np.asarray(successes, dtype=bool)
study_attempts = study_successes.shape[1]
if study_attempts < 32:
    raise ValueError(
        "Solvability study needs at least 32 recorded attempts per puzzle. "
        "Rerun the evaluation cell with ATTEMPTS_PER_PUZZLE = 32."
    )
study_rates = study_successes.mean(axis=1)
study_probabilities_percent = 100.0 * study_rates

SOLVABILITY_BANDWIDTH_PP = 100.0 / study_attempts
SOLVABILITY_DENSITY_FLOOR = 0.001
if SOLVABILITY_BANDWIDTH_PP <= 0 or SOLVABILITY_DENSITY_FLOOR <= 0:
    raise ValueError("Bandwidth and the log-display density floor must be positive.")
solvability_grid = np.linspace(0.0, 100.0, 2001)
bandwidth = float(SOLVABILITY_BANDWIDTH_PP)
distance = (
    solvability_grid[:, None] - study_probabilities_percent[None, :]
) / bandwidth
left_reflection = (
    solvability_grid[:, None] + study_probabilities_percent[None, :]
) / bandwidth
right_reflection = (
    solvability_grid[:, None] - (200.0 - study_probabilities_percent[None, :])
) / bandwidth
solvability_density = (
    np.exp(-0.5 * distance**2)
    + np.exp(-0.5 * left_reflection**2)
    + np.exp(-0.5 * right_reflection**2)
).mean(axis=1) / (bandwidth * np.sqrt(2.0 * np.pi))
# Reflection retains boundary mass at 0% and 100%; normalize the sampled grid.
density_area = np.sum(
    0.5
    * (solvability_density[:-1] + solvability_density[1:])
    * np.diff(solvability_grid)
)
solvability_density_percent = 100.0 * solvability_density / density_area
visible_density = np.where(
    solvability_density_percent >= SOLVABILITY_DENSITY_FLOOR,
    solvability_density_percent,
    np.nan,
)

# Pick distinct examples: highest observed rate, then approximately half
# the previous selected example's rate. Use actual available measurements.
study_example_positions = []
study_example_targets = []
remaining_positions = np.arange(len(study_rates))
previous_rate = None
for _ in range(min(3, len(study_rates))):
    if previous_rate is None:
        target_rate = float(study_rates.max())
        candidates = remaining_positions
    else:
        target_rate = previous_rate / 2.0
        lower_candidates = remaining_positions[
            study_rates[remaining_positions] < previous_rate
        ]
        candidates = lower_candidates if len(lower_candidates) else remaining_positions
    differences = np.abs(study_rates[candidates] - target_rate)
    position = int(candidates[np.argmin(differences)])
    study_example_positions.append(position)
    study_example_targets.append(target_rate)
    previous_rate = float(study_rates[position])
    remaining_positions = remaining_positions[remaining_positions != position]

study_example_colors = ["#D88437", "#2F9D88", "#9B6BD3"]
solvability_fig = go.Figure(
    go.Scatter(
        x=solvability_grid,
        y=visible_density,
        mode="lines",
        line=dict(color="#4F6BED", width=3),
        name="Smoothed measured solvability",
        showlegend=False,
        hovertemplate=(
            "Probability to be solved: %{x:.2f}%"
            "<br>Puzzles per 1 percentage point: %{y:.4f}%<extra></extra>"
        ),
    )
)
for rank, position in enumerate(study_example_positions):
    probability_percent = float(study_probabilities_percent[position])
    density_at_probability = float(
        np.interp(probability_percent, solvability_grid, solvability_density_percent)
    )
    solvability_fig.add_vline(
        x=probability_percent,
        line=dict(color=study_example_colors[rank], width=1.3, dash="dot"),
    )
    solvability_fig.add_trace(
        go.Scatter(
            x=[probability_percent],
            y=[density_at_probability],
            mode="markers",
            marker=dict(color=study_example_colors[rank], size=10),
            name=f"Example {rank + 1}: {probability_percent:.2f}%",
            hovertemplate=(
                f"Example {rank + 1}<br>Measured solvability: {probability_percent:.2f}%"
                "<extra></extra>"
            ),
        )
    )
solvability_fig.update_layout(
    title=dict(
        text=f"Solvability study - {len(study_rates)} puzzles, "
        f"{study_attempts} attempts each",
        x=0,
    ),
    template="plotly_white",
    autosize=True,
    height=500,
    xaxis_title="Probability to be solved (%)",
    yaxis_title="% of puzzles per 1 percentage point (log scale)",
    yaxis_type="log",
    legend=dict(orientation="h", x=0, y=-0.24),
    margin=dict(l=85, r=25, t=85, b=115),
    font=dict(family="Arial, sans-serif", size=16),
)
solvability_fig.update_xaxes(range=[0, 100], dtick=10)
solvability_fig.update_yaxes(
    range=[
        np.log10(SOLVABILITY_DENSITY_FLOOR),
        np.ceil(np.log10(solvability_density_percent.max())) + 0.1,
    ]
)
solvability_fig.show(config=PLOT_CONFIG)
display(
    HTML(
        f"Measured resolution: {100 / study_attempts:.3f} percentage points per success."
        f" Smoothing bandwidth: {bandwidth:.3f} percentage points.<br>"
        f"No observed success: {np.mean(study_rates == 0):.1%} of puzzles."
    )
)

np.savez_compressed(
    ARTIFACTS / "queens7_solvability_study.npz",
    dataset_indices=evaluation_indices,
    successes=study_successes,
    per_puzzle_probability=study_rates,
    grid_probability_percent=solvability_grid,
    density_percent_per_percentage_point=solvability_density_percent,
    bandwidth_percentage_points=bandwidth,
    example_positions=np.asarray(study_example_positions),
    dataset_sha256=puzzles["metadata"]["data_sha256"],
)

### 7.3 Example puzzles by solvability

Start with the most frequently solved evaluated puzzle. Then select a different puzzle whose measured success probability is closest to half of the previous example's probability, preferring a strictly lower rate when one is available.

The examples appear in decreasing solvability order. Titles show the **actual measured probability and success count**, since exact 100%, 50% and 25% examples may not exist. The queens shown are the supplied ground-truth solution on the original coloring.


In [17]:
example_titles = []
for rank, position in enumerate(study_example_positions):
    count = int(study_successes[position].sum())
    probability_percent = float(study_probabilities_percent[position])
    example_titles.append(
        f"Example {rank + 1}: {probability_percent:.2f}%"
        f"<br>{count}/{study_attempts} successful attempts"
    )
study_examples_fig = make_subplots(
    rows=1,
    cols=len(study_example_positions),
    subplot_titles=example_titles,
    horizontal_spacing=0.065,
)
selected_study_examples = []
for col, position in enumerate(study_example_positions, start=1):
    puzzle_index = int(evaluation_indices[position])
    add_board(study_examples_fig, regions[puzzle_index], regions[puzzle_index], 1, col)
    queen_rows, queen_cols = np.where(queen_masks[puzzle_index])
    study_examples_fig.add_trace(
        go.Scatter(
            x=queen_cols,
            y=queen_rows,
            mode="text",
            text=[chr(0x265B)] * len(queen_rows),
            textfont=dict(size=25, color="#172033"),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=1,
        col=col,
    )
    selected_study_examples.append(
        {
            "dataset_index": puzzle_index,
            "id": puzzles["source_ids"][puzzle_index],
            "source": puzzles["source_names"][puzzle_index],
            "successes": int(study_successes[position].sum()),
            "attempts": study_attempts,
            "probability": float(study_rates[position]),
            "target_probability": float(study_example_targets[col - 1]),
        }
    )
style_boards(
    study_examples_fig,
    "Solvability study - example puzzles",
    rows=1,
)
study_examples_fig.update_layout(height=410, margin=dict(l=10, r=10, t=120, b=20))
study_examples_fig.show(config=PLOT_CONFIG)
display(
    HTML(
        "<br>".join(
            f"<b>Example {rank + 1}</b>: "
            f"{html.escape(str(example['source']))} / {html.escape(str(example['id']))}"
            for rank, example in enumerate(selected_study_examples)
        )
    )
)

## 8. DFM solvability and deduction-only solving

For each evaluated test puzzle, compare:

- Its measured DFM success probability $\widehat p_i=c_i/A$, using the
  $A$ already recorded attempts (32 by default).
- Whether [gitars/queens-generator](https://github.com/gitars/queens-generator)
  completes it with the repository's **ordered human-style deduction sequence**.

The solver receives only the region map and an initially unmarked board.
For each puzzle we call the upstream <code>speed_stars.deductions.generate_deductions</code>,
then replay its returned <code>star</code> and <code>crossedOut</code> steps from
a blank board and check the final grid. This sequence includes forced cells,
subset elimination, cell-by-cell deductions, X-Wing and the repository's
one-level lookahead rule. It does not recursively branch until it finds a
complete solution. **Not solved deductively** means this deduction generator
stalled before completing the board; it does not mean every possible
deduction system would fail.
The dependency is pinned to commit
<code>74bdd520eff149384afa3dd4c055b44e64ce0394</code>.

### Reading the rectangle

Each column groups puzzles with the **same measured DFM probability** $k/A$.
The column height is 100% of that group:

$$
g_k=
\frac{\#\{i:c_i=k\;\text{and deduction solved }i\}}
     {\#\{i:c_i=k\}},\qquad
\text{green}=100g_k,\quad\text{red}=100(1-g_k).
$$

Red is at the bottom and green at the top. Hover to see group counts.
Gray columns have no observations. These are empirical fractions without
smoothing; each observed success changes $\widehat p_i$ by $1/A$.

The next cell runs the deduction generator, reuses cached verdicts where available, and reads the current DFM evaluation if present; otherwise
<code>artifacts/queens7/queens7_dfm_evaluation.npz</code>.
It checks that all IDs belong to the current test split and that a loaded
cache matches the dataset. The comparison covers exactly those evaluated
puzzles. To cover the entire test split, first run the DFM evaluation with
<code>N_EVAL_PUZZLES = None</code>.

Deductive verdicts and their ordered step traces are cached by puzzle
fingerprint. The cell also writes per-puzzle DFM probabilities, deduction
verdicts and step traces to JSONL/CSV, and displays
an expandable list below the chart. No model evaluation or puzzle generation
is performed here.


In [19]:
import csv
import logging
from speed_stars.core import (
    CROSSED_OUT_CELL_VALUE as DEDUCTION_CROSSED,
    EMPTY_CELL_VALUE as DEDUCTION_EMPTY,
    STAR_CELL_VALUE as DEDUCTION_QUEEN,
    cross_out_if_empty as deduction_cross_out,
    is_solved as upstream_is_solved,
    place_star as deduction_place_star,
)
from speed_stars.deductions import generate_deductions

DEDUCTION_SOLVER_COMMIT = "74bdd520eff149384afa3dd4c055b44e64ce0394"
DEDUCTION_SOLVER_MODE = "generate_deductions_v1"
DEDUCTION_CACHE_PATH = ARTIFACTS / "queens7_deductive_verdicts.json"
DEDUCTION_RECORDS_PATH = ARTIFACTS / "queens7_dfm_deductive_comparison.jsonl"
DEDUCTION_CSV_PATH = ARTIFACTS / "queens7_dfm_deductive_comparison.csv"
DEDUCTION_RESULTS_PATH = ARTIFACTS / "queens7_dfm_deductive_comparison.npz"
deduction_dataset_sha = puzzles["metadata"]["data_sha256"]

if "evaluation_results" in globals():
    deduction_evaluation = evaluation_results
    deduction_evaluation_origin = "Current DFM evaluation"
else:
    deduction_evaluation_path = ARTIFACTS / "queens7_dfm_evaluation.npz"
    if not deduction_evaluation_path.exists():
        raise FileNotFoundError(
            "Run the DFM held-out evaluation first; no saved evaluation cache exists."
        )
    with np.load(deduction_evaluation_path, allow_pickle=False) as saved:
        deduction_evaluation = {key: saved[key].copy() for key in saved.files}
    if str(deduction_evaluation["dataset_sha256"]) != deduction_dataset_sha:
        raise ValueError("The saved DFM evaluation uses a different dataset.")
    deduction_evaluation_origin = "Saved DFM evaluation"

deduction_indices = np.asarray(deduction_evaluation["dataset_indices"], dtype=np.int64)
deduction_attempt_successes = np.asarray(deduction_evaluation["successes"], dtype=bool)
if (
    deduction_indices.ndim != 1
    or not len(deduction_indices)
    or len(np.unique(deduction_indices)) != len(deduction_indices)
    or not np.isin(deduction_indices, splits["test"]).all()
):
    raise ValueError(
        "Expected distinct evaluated puzzle IDs from the current test split."
    )
if (
    deduction_attempt_successes.ndim != 2
    or deduction_attempt_successes.shape[0] != len(deduction_indices)
    or deduction_attempt_successes.shape[1] < 1
):
    raise ValueError("Expected a nonempty (puzzles, attempts) DFM success array.")
deduction_attempt_count = deduction_attempt_successes.shape[1]
deduction_dfm_counts = deduction_attempt_successes.sum(axis=1)
deduction_dfm_probabilities = deduction_attempt_successes.mean(axis=1)

deduction_cache = {}
if DEDUCTION_CACHE_PATH.exists():
    saved_deductions = json.loads(DEDUCTION_CACHE_PATH.read_text(encoding="utf-8"))
    if (
        saved_deductions.get("dataset_sha256") == deduction_dataset_sha
        and saved_deductions.get("solver_commit") == DEDUCTION_SOLVER_COMMIT
        and saved_deductions.get("solver_mode") == DEDUCTION_SOLVER_MODE
    ):
        deduction_cache = saved_deductions.get("puzzles", {})

deduction_progress = widgets.IntProgress(
    value=0,
    max=len(deduction_indices),
    description="Deduction solving",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
deduction_status = widgets.HTML()
display(deduction_progress, deduction_status)
deduction_records = []
deduction_partial_grids = []
deduction_traces = []
deduction_cache_hits = 0
for position, puzzle_index in enumerate(deduction_indices):
    puzzle_index = int(puzzle_index)
    fingerprint = str(puzzles["fingerprints"][puzzle_index])
    board = regions[puzzle_index]
    if fingerprint in deduction_cache:
        verdict = deduction_cache[fingerprint]
        deduction_cache_hits += 1
    else:
        zones = board.tolist()
        partial = [[DEDUCTION_EMPTY] * BOARD_SIZE for _ in range(BOARD_SIZE)]
        # The upstream hint engine logs when it reaches lookahead or stalls;
        # keep those per-puzzle diagnostics in the structured verdicts instead.
        previous_logging_disable = logging.root.manager.disable
        logging.disable(logging.CRITICAL)
        try:
            deductions = generate_deductions(zones)
        finally:
            logging.disable(previous_logging_disable)
        for deduction in deductions:
            for cell in deduction.cells:
                if deduction.state == "star":
                    partial = deduction_place_star(partial, zones, cell.x, cell.y)
                elif deduction.state == "crossedOut":
                    deduction_cross_out(partial, cell.x, cell.y)
                else:
                    raise ValueError(f"Unknown deduction state: {deduction.state}")
        solved = bool(upstream_is_solved(partial, zones))
        deduction_trace = [
            dict(
                state=item.state,
                cells=[[cell.x, cell.y] for cell in item.cells],
                reason=item.reason,
            )
            for item in deductions
        ]
        # Check a reported solution against exactly the same rules used for DFM.
        # This is validation of the returned board, never a search or a target hint.
        if solved:
            tokens = np.where(np.asarray(partial) == DEDUCTION_QUEEN, board, EMPTY)
            if not bool(solution_checks(board, tokens)["solved"]):
                raise RuntimeError(
                    f"Upstream solver returned an invalid solution: {puzzle_index}."
                )
        verdict = dict(
            is_solved_deductively=solved,
            status="solved" if solved else "stalled",
            deduction_steps=deduction_trace,
            partial_grid=partial,
            queens_placed=int(np.sum(np.asarray(partial) == DEDUCTION_QUEEN)),
            undecided_cells=int(np.sum(np.asarray(partial) == DEDUCTION_EMPTY)),
        )
        deduction_cache[fingerprint] = verdict
    deduction_partial_grids.append(verdict["partial_grid"])
    deduction_traces.append(verdict["deduction_steps"])
    deduction_records.append(
        dict(
            dataset_index=puzzle_index,
            fingerprint=fingerprint,
            source=str(puzzles["source_names"][puzzle_index]),
            source_id=str(puzzles["source_ids"][puzzle_index]),
            probability_to_be_solved_via_diffusion=float(
                deduction_dfm_probabilities[position]
            ),
            diffusion_successes=int(deduction_dfm_counts[position]),
            diffusion_attempts=int(deduction_attempt_count),
            is_solved_deductively=bool(verdict["is_solved_deductively"]),
            deduction_status=verdict["status"],
            deduction_queens_placed=int(verdict["queens_placed"]),
            deduction_steps=verdict["deduction_steps"],
            deduction_undecided_cells=int(verdict["undecided_cells"]),
        )
    )
    deduction_progress.value = position + 1
    deduction_status.value = (
        f"{position + 1:,} / {len(deduction_indices):,} puzzles · "
        f"{deduction_cache_hits:,} cached · last verdict: {verdict['status']}"
    )
deduction_progress.bar_style = "success"
DEDUCTION_CACHE_PATH.write_text(
    json.dumps(
        dict(
            dataset_sha256=deduction_dataset_sha,
            solver_commit=DEDUCTION_SOLVER_COMMIT,
            solver_mode=DEDUCTION_SOLVER_MODE,
            puzzles=deduction_cache,
        ),
        ensure_ascii=False,
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)
DEDUCTION_RECORDS_PATH.write_text(
    "".join(
        json.dumps(record, ensure_ascii=False) + "\n" for record in deduction_records
    ),
    encoding="utf-8",
)
with DEDUCTION_CSV_PATH.open("w", encoding="utf-8-sig", newline="") as output:
    writer = csv.DictWriter(output, fieldnames=list(deduction_records[0]))
    writer.writeheader()
    writer.writerows(deduction_records)

deduction_solved = np.asarray(
    [record["is_solved_deductively"] for record in deduction_records], dtype=bool
)
deduction_support = np.arange(deduction_attempt_count + 1)
deduction_group_counts = np.bincount(
    deduction_dfm_counts,
    minlength=len(deduction_support),
)
deduction_group_solved = np.bincount(
    deduction_dfm_counts[deduction_solved],
    minlength=len(deduction_support),
)
deduction_group_unsolved = deduction_group_counts - deduction_group_solved
deduction_group_solved_percent = np.divide(
    100.0 * deduction_group_solved,
    deduction_group_counts,
    out=np.zeros(len(deduction_support), dtype=float),
    where=deduction_group_counts > 0,
)
deduction_group_unsolved_percent = np.divide(
    100.0 * deduction_group_unsolved,
    deduction_group_counts,
    out=np.zeros(len(deduction_support), dtype=float),
    where=deduction_group_counts > 0,
)

# Columns cover nearest measured probabilities, with half-width boundary columns.
# Their widths represent probability intervals, not group sample sizes.
deduction_support_percent = 100.0 * deduction_support / deduction_attempt_count
deduction_edges = np.concatenate(
    (
        [0.0],
        (deduction_support_percent[:-1] + deduction_support_percent[1:]) / 2,
        [100.0],
    )
)
deduction_column_centers = (deduction_edges[:-1] + deduction_edges[1:]) / 2
deduction_column_widths = np.diff(deduction_edges)
deduction_present = deduction_group_counts > 0
deduction_customdata = np.column_stack(
    (
        deduction_support_percent,
        deduction_group_counts,
        deduction_group_solved,
        deduction_group_unsolved,
        deduction_support,
    )
)
deduction_figure = go.Figure()
# Add red first so green stacks above it.
for percentages, name, color, rank in (
    (deduction_group_unsolved_percent, "Not solved deductively", "#DC6C69", 1),
    (deduction_group_solved_percent, "Solved deductively", "#42A889", 0),
):
    deduction_figure.add_trace(
        go.Bar(
            x=deduction_column_centers[deduction_present],
            width=deduction_column_widths[deduction_present],
            y=percentages[deduction_present],
            name=name,
            legendrank=rank,
            marker=dict(color=color, line=dict(width=0)),
            customdata=deduction_customdata[deduction_present],
            hovertemplate=(
                "DFM success probability: %{customdata[0]:.3f}%"
                f" (%{{customdata[4]:.0f}}/{deduction_attempt_count})"
                "<br>This category: %{y:.1f}%"
                "<br>Puzzles in group: %{customdata[1]:.0f}"
                "<br>Solved deductively: %{customdata[2]:.0f}"
                "<br>Not solved deductively: %{customdata[3]:.0f}"
                "<extra>%{fullData.name}</extra>"
            ),
        )
    )
deduction_figure.update_layout(
    title=dict(text="Deductive solvability vs DFM success probability", x=0),
    template="plotly_white",
    plot_bgcolor="#EEF1F4",
    autosize=True,
    height=410,
    barmode="stack",
    bargap=0,
    bargroupgap=0,
    xaxis=dict(
        title="Probability to be solved via diffusion (%)",
        range=[0, 100],
        dtick=10,
        showgrid=False,
        zeroline=False,
    ),
    yaxis=dict(
        title="Puzzles within each probability group (%)",
        range=[0, 100],
        dtick=25,
        showgrid=False,
        zeroline=False,
    ),
    legend=dict(orientation="h", x=0, y=1.12, traceorder="normal"),
    margin=dict(l=85, r=20, t=105, b=65),
    font=dict(family="Arial, sans-serif", size=16),
)
deduction_figure.show(config=PLOT_CONFIG)
deduction_stalled = sum(
    record["deduction_status"] == "stalled" for record in deduction_records
)
display(
    HTML(
        f"<b>Deductively solved:</b> {int(deduction_solved.sum())}/"
        f"{len(deduction_solved)} ({100 * deduction_solved.mean():.1f}%) · "
        f"{deduction_attempt_count} DFM attempts per puzzle · "
        f"measured probability spacing {100 / deduction_attempt_count:.3f} percentage points"
        f"<br>{deduction_evaluation_origin}. Gray columns: no puzzles. "
        f"Stalled before completion: {deduction_stalled}."
    )
)

deduction_verdict_rows = []
for record in sorted(
    deduction_records,
    key=lambda item: (
        item["probability_to_be_solved_via_diffusion"],
        item["dataset_index"],
    ),
):
    color = "#278065" if record["is_solved_deductively"] else "#B74F4C"
    deduction_verdict_rows.append(
        f"<div style='padding:4px 0;border-bottom:1px solid #edf0f3'>"
        f"<b>#{record['dataset_index']}</b> · "
        f"{html.escape(record['source'])} / {html.escape(record['source_id'])} · "
        f"DFM: {100 * record['probability_to_be_solved_via_diffusion']:.3f}% "
        f"({record['diffusion_successes']}/{record['diffusion_attempts']}) · "
        f"<span style='color:{color}' title='{html.escape('; '.join(step["reason"] for step in record["deduction_steps"]))}'>{record['deduction_status']} ? {len(record['deduction_steps'])} steps</span></div>"
    )
display(
    HTML(
        "<details><summary>Per-puzzle probabilities and deduction verdicts</summary>"
        "<div style='max-height:300px;overflow-y:auto;margin-top:10px'>"
        + "".join(deduction_verdict_rows)
        + "</div></details>"
        f"<p>Per-puzzle records: <code>{DEDUCTION_RECORDS_PATH.name}</code>, "
        f"<code>{DEDUCTION_CSV_PATH.name}</code> in <code>artifacts/queens7/</code>.</p>"
    )
)
deduction_comparison_results = dict(
    dataset_indices=deduction_indices,
    per_puzzle_diffusion_probability=deduction_dfm_probabilities,
    diffusion_successes=deduction_attempt_successes,
    is_solved_deductively=deduction_solved,
    deduction_status=np.asarray(
        [record["deduction_status"] for record in deduction_records]
    ),
    deduction_partial_grids=np.asarray(deduction_partial_grids, dtype=np.int8),
    deduction_traces_json=np.asarray(
        [json.dumps(trace, ensure_ascii=False) for trace in deduction_traces]
    ),
    group_probability=deduction_support_percent / 100,
    group_counts=deduction_group_counts,
    group_solved_counts=deduction_group_solved,
    group_solved_percent=deduction_group_solved_percent,
    dataset_sha256=deduction_dataset_sha,
    solver_commit=DEDUCTION_SOLVER_COMMIT,
    solver_mode=DEDUCTION_SOLVER_MODE,
    diffusion_steps=int(deduction_evaluation["steps"]),
)
np.savez_compressed(DEDUCTION_RESULTS_PATH, **deduction_comparison_results)

IntProgress(value=0, description='Deduction solving', layout=Layout(width='100%'), max=256, style=ProgressStyl…

HTML(value='')

### 8.1 A trace metric: mean eliminations before placing a queen

The solved/unsolved deduction verdict is constant in the current 256-puzzle
sample: all deduction traces finish. The traces still differ in how much
reasoning precedes their queen placements.

I inspected traces across DFM probability groups. Puzzles below 25% DFM
success had about 14.8 recorded deductions on average, versus 11.3 for
puzzles at or above 75%. Merely detecting lookahead is too coarse: it appears
in few traces. A useful simple measure is **how many elimination deductions
each queen had to wait for**.

For a list of deductions $D=(d_1,\ldots,d_L)$, let $Q$ be the total number
of queen placements. For each queen $q$, count the preceding deductions whose
state is <code>crossedOut</code>:

$$
E_q=\#\{s:\ d_s.\mathrm{state}=\texttt{crossedOut}
                  \text{ and }s\text{ precedes placement of queen }q\}.
$$

Define the metric

$$
M(D)=\frac{1}{Q}\sum_{q=1}^{Q}E_q.
$$

For example, the sequence
<code>exclude → queen → exclude → exclude → queen</code>
has $E=(1,3)$ and $M=2$.

An equivalent expression explains why early eliminations contribute more:

$$
M(D)=\frac{1}{Q}\sum_{s:\,d_s.\mathrm{state}=\texttt{crossedOut}}R_s,
$$

where $R_s$ is the number of queens not yet placed before deduction $s$.
A batch of excluded cells counts as **one explanation**, regardless of its size.
Queen placements and their automatic cross-outs do not add elimination steps.
The metric uses only the deduction list, without the model's outcomes,
the board geometry, fitted weights or source labels.

**Interpretation:** larger $M$ means that queen placements are delayed behind
more explicit elimination reasoning. This is a property of this solver's trace
and rule order, not a minimum proof length. Lookahead's internal propagation
is absent from the returned list, so $M$ does not measure its full computational cost.

The scatter below places one puzzle at
$(M(D_i),100\,c_i/A)$, colored by dataset source. The black curve shows mean
DFM success within up to five equal-count groups of the metric, with 95%
puzzle-bootstrap intervals. It is an empirical calibration, not a fitted
predictor. Higher metrics are expected to correspond to lower DFM solvability,
but individual puzzles can deviate substantially.

The initial inspection found a moderate negative association (Spearman
approximately $-0.41$). Source-specific associations are also reported because
sources differ in difficulty. These observations helped choose the metric on
the current test results: they are exploratory, not an independent estimate
of prediction quality on new puzzles.

Run this cell after dataset setup and the deduction comparison, or reuse their
saved artifacts. It joins the traces to the current DFM evaluation by puzzle
fingerprint and dataset index. No model calls or new deduction solving are needed.


In [ ]:
import csv


def deduction_wait_metric(deductions):
    '''Mean number of explicit elimination steps before each queen placement.

    Accepts upstream Deduction objects or their saved dictionary representation.
    '''
    def field(step, name):
        return step[name] if isinstance(step, dict) else getattr(step, name)

    eliminations = 0
    queen_waits = []
    for step in deductions:
        state = field(step, "state")
        if state == "crossedOut":
            eliminations += 1
        elif state == "star":
            queen_waits.extend([eliminations] * len(field(step, "cells")))
        else:
            raise ValueError(f"Unknown deduction state: {state}")
    if not queen_waits:
        raise ValueError("A trace must contain at least one queen placement.")
    return float(np.mean(queen_waits)), queen_waits


def average_ranks(values):
    _, inverse, counts = np.unique(values, return_inverse=True, return_counts=True)
    starts = np.cumsum(counts) - counts
    return (starts + (counts - 1) / 2)[inverse]


def trace_metric_spearman(x, y):
    x, y = average_ranks(np.asarray(x)), average_ranks(np.asarray(y))
    if len(x) < 2 or np.std(x) == 0 or np.std(y) == 0:
        return float("nan")
    return float(np.corrcoef(x, y)[0, 1])


if "evaluation_results" in globals():
    metric_evaluation = evaluation_results
else:
    with np.load(ARTIFACTS / "queens7_dfm_evaluation.npz", allow_pickle=False) as saved:
        metric_evaluation = {key: saved[key].copy() for key in saved.files}
if (
    "dataset_sha256" in metric_evaluation
    and str(metric_evaluation["dataset_sha256"]) != puzzles["metadata"]["data_sha256"]
):
    raise ValueError("DFM evaluation and current dataset do not match.")
metric_indices = np.asarray(metric_evaluation["dataset_indices"], dtype=np.int64)
metric_attempts = np.asarray(metric_evaluation["successes"], dtype=bool)
if (
    metric_indices.ndim != 1 or not len(metric_indices)
    or len(np.unique(metric_indices)) != len(metric_indices)
    or not np.isin(metric_indices, splits["test"]).all()
    or metric_attempts.ndim != 2 or metric_attempts.shape[0] != len(metric_indices)
    or metric_attempts.shape[1] < 1
):
    raise ValueError("Expected DFM attempts aligned with distinct current test puzzle IDs.")

if "deduction_records" in globals():
    metric_trace_records = deduction_records
else:
    metric_trace_path = ARTIFACTS / "queens7_dfm_deductive_comparison.jsonl"
    if not metric_trace_path.exists():
        raise FileNotFoundError("Run the deduction-solving cell in Section 8 first.")
    metric_trace_records = [
        json.loads(line) for line in metric_trace_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
metric_trace_lookup = {
    record["fingerprint"]: record for record in metric_trace_records
}
deduction_metric_records = []
metric_excluded_indices = []
for position, index in enumerate(metric_indices):
    index = int(index)
    fingerprint = str(puzzles["fingerprints"][index])
    record = metric_trace_lookup.get(fingerprint)
    if record is None or "deduction_steps" not in record:
        raise ValueError(f"Missing deduction trace for puzzle {index}; rerun Section 8.")
    # Incomplete traces would make the mean over already placed queens biased.
    if not record["is_solved_deductively"]:
        metric_excluded_indices.append(index)
        continue
    metric, waits = deduction_wait_metric(record["deduction_steps"])
    if len(waits) != BOARD_SIZE:
        raise ValueError(f"Puzzle {index} does not have a complete queen-placement trace.")
    successes_count = int(metric_attempts[position].sum())
    lookahead_count = sum(
        step["reason"] in ("This position cannot be a star", "This position must be a star")
        for step in record["deduction_steps"]
    )
    deduction_metric_records.append(dict(
        dataset_index=index, fingerprint=fingerprint,
        source=str(puzzles["source_names"][index]),
        source_id=str(puzzles["source_ids"][index]),
        mean_pre_queen_eliminations=metric,
        queen_waits=waits,
        trace_length=len(record["deduction_steps"]),
        explicit_elimination_steps=sum(
            step["state"] == "crossedOut" for step in record["deduction_steps"]
        ),
        lookahead_steps=int(lookahead_count),
        diffusion_successes=successes_count,
        diffusion_attempts=int(metric_attempts.shape[1]),
        diffusion_solvability=successes_count / metric_attempts.shape[1],
    ))
if not deduction_metric_records:
    raise ValueError("No complete deduction traces are available for this comparison.")

metric_values = np.asarray([
    row["mean_pre_queen_eliminations"] for row in deduction_metric_records
])
metric_rates = np.asarray([row["diffusion_solvability"] for row in deduction_metric_records])
metric_sources = np.asarray([row["source"] for row in deduction_metric_records])
metric_rho = trace_metric_spearman(metric_values, metric_rates)
metric_source_colors = {
    "Synthetic": "#5267DB", "Queens.game": "#D88437",
    "AMAZE": "#2F9D88", "Queendom": "#9B6BD3",
}
metric_figure = go.Figure()
for source in sorted(set(metric_sources.tolist())):
    positions = np.flatnonzero(metric_sources == source)
    custom = [[
        str(deduction_metric_records[i]["dataset_index"]),
        deduction_metric_records[i]["source_id"],
        str(deduction_metric_records[i]["diffusion_successes"]),
        str(deduction_metric_records[i]["diffusion_attempts"]),
        str(deduction_metric_records[i]["queen_waits"]),
        str(deduction_metric_records[i]["trace_length"]),
        str(deduction_metric_records[i]["lookahead_steps"]),
    ] for i in positions]
    metric_figure.add_trace(go.Scatter(
        x=metric_values[positions], y=100 * metric_rates[positions],
        mode="markers", name=f"{source} · n={len(positions)}",
        marker=dict(size=9, opacity=0.65, color=metric_source_colors.get(source, "#64748B")),
        customdata=custom,
        hovertemplate=(
            "Puzzle %{customdata[0]} · %{customdata[1]}"
            "<br>Mean pre-queen eliminations: %{x:.3f}"
            "<br>DFM solvability: %{y:.3f}% (%{customdata[2]}/%{customdata[3]})"
            "<br>Wait before each queen: %{customdata[4]}"
            "<br>Trace length: %{customdata[5]} · lookahead steps: %{customdata[6]}"
            "<extra>%{fullData.name}</extra>"
        ),
    ))

# Descriptive calibration by metric quintiles; the metric itself has no fitted parameters.
metric_edges = np.unique(np.quantile(metric_values, np.linspace(0, 1, 6)))
metric_groups = np.searchsorted(metric_edges[1:-1], metric_values, side="right")
metric_calibration = []
metric_bootstrap_rng = np.random.default_rng(SEED + 804)
for group in np.unique(metric_groups):
    selected = metric_groups == group
    values = metric_rates[selected]
    draws = metric_bootstrap_rng.integers(0, len(values), size=(2000, len(values)))
    low, high = np.quantile(values[draws].mean(axis=1), [0.025, 0.975])
    metric_calibration.append(dict(
        mean_metric=float(metric_values[selected].mean()),
        mean_probability=float(values.mean()),
        lower_probability=float(low), upper_probability=float(high),
        count=int(selected.sum()),
        min_metric=float(metric_values[selected].min()),
        max_metric=float(metric_values[selected].max()),
    ))
metric_figure.add_trace(go.Scatter(
    x=[row["mean_metric"] for row in metric_calibration],
    y=[100 * row["mean_probability"] for row in metric_calibration],
    mode="lines+markers", name="Mean within metric groups · 95% interval",
    line=dict(color="#172033", width=3), marker=dict(size=10, symbol="diamond"),
    error_y=dict(
        type="data", symmetric=False,
        array=[100 * (row["upper_probability"] - row["mean_probability"])
               for row in metric_calibration],
        arrayminus=[100 * (row["mean_probability"] - row["lower_probability"])
                    for row in metric_calibration],
    ),
    customdata=[[row["count"], row["min_metric"], row["max_metric"]]
                for row in metric_calibration],
    hovertemplate=(
        "Metric group: [%{customdata[1]:.2f}, %{customdata[2]:.2f}]"
        "<br>%{customdata[0]} puzzles · mean metric %{x:.2f}"
        "<br>Mean DFM solvability %{y:.2f}%<extra></extra>"
    ),
))
metric_rho_text = f"{metric_rho:+.3f}" if np.isfinite(metric_rho) else "undefined"
metric_figure.update_layout(
    title=dict(
        text=f"Mean pre-queen eliminations vs DFM solvability"
             f"<br><sup>{len(metric_values)} puzzles · Spearman ρ = {metric_rho_text}</sup>",
        x=0,
    ),
    template="plotly_white", autosize=True, height=560,
    xaxis=dict(title="Mean elimination deductions before placing a queen · M(D)",
               rangemode="tozero"),
    yaxis=dict(title="Probability to be solved via diffusion (%)",
               range=[-2, 102], dtick=20),
    legend=dict(orientation="h", x=0, y=-0.24),
    margin=dict(l=75, r=25, t=100, b=140),
    font=dict(family="Arial, sans-serif", size=16),
)
metric_figure.show(config=PLOT_CONFIG)
metric_source_statistics = []
for source in sorted(set(metric_sources.tolist())):
    selected = metric_sources == source
    rho = trace_metric_spearman(metric_values[selected], metric_rates[selected])
    metric_source_statistics.append(dict(
        source=source, count=int(selected.sum()),
        spearman=None if not np.isfinite(rho) else rho,
    ))
display(HTML(
    "<b>Spearman within each source:</b> "
    + " · ".join(
        f"{html.escape(row['source'])}: "
        + (f"{row['spearman']:+.3f}" if row["spearman"] is not None else "undefined")
        + f" (n={row['count']})"
        for row in metric_source_statistics
    )
    + f"<br>Incomplete traces excluded: {len(metric_excluded_indices)}. "
    "This is exploratory association on the current model's test results."
))
metric_artifact = dict(
    definition="mean_pre_queen_eliminations_v1",
    dataset_sha256=puzzles["metadata"]["data_sha256"],
    diffusion_steps=int(metric_evaluation["steps"]),
    spearman=None if not np.isfinite(metric_rho) else metric_rho,
    source_statistics=metric_source_statistics,
    calibration_groups=metric_calibration,
    excluded_incomplete_indices=metric_excluded_indices,
    puzzles=deduction_metric_records,
)
(ARTIFACTS / "queens7_deduction_wait_metric.json").write_text(
    json.dumps(metric_artifact, ensure_ascii=False, indent=2) + "\n", encoding="utf-8",
)
with (ARTIFACTS / "queens7_deduction_wait_metric.csv").open(
    "w", encoding="utf-8-sig", newline="",
) as output:
    writer = csv.DictWriter(output, fieldnames=list(deduction_metric_records[0]))
    writer.writeheader()
    writer.writerows(deduction_metric_records)
metric_figure.write_html(
    ARTIFACTS / "queens7_deduction_wait_metric.html",
    include_plotlyjs=True, config=PLOT_CONFIG,
)


### 8.2 Queen → automatic exclusions → queen: dependency depth

The waiting metric omits automatic exclusions made by a placed queen.
Here we reconstruct a graph of those dependencies from the **region map and
recorded deduction trace**, then compare its depth with DFM solvability.

At the time a queen $q$ is placed, let $E_q$ be the cells explicitly marked
<code>crossedOut</code> by earlier deductions. For a subset $S$ of earlier
queens, let $A(S)$ be the cells their placements exclude: the same row,
column, region, and adjacent cells.

Queen $q$ is forced in a row, column or region $U$ if all its alternatives
are excluded:

$$
U\setminus\{q\}\ \subseteq\ E_q\cup A(S).
$$

Find supporting earlier queens $S$ and a unit $U$ minimizing the resulting
dependency depth:

$$
d(q)=1+\min_{\substack{S,U:\\U\setminus\{q\}\subseteq E_q\cup A(S)}}
           \max_{p\in S}d(p),\qquad \max\varnothing=0.
$$

A queen forced by explicit exclusions alone has depth 1. A queen requiring
one of those queens has depth 2, and so on. Ties prefer fewer parents.
Draw an arrow $p\to q$ for each selected supporting queen.
For seven queens there are at most 64 subsets of earlier queens to inspect
for a placement; this inspects supports in an already solved trace.

The primary metric is the longest chain:

$$
D_{\max}=\max_q d(q).
$$

Also report $D_{\mathrm{mean}}=\frac17\sum_q d(q)$ and the previous
waiting metric $M$ for comparison. No DFM outcome enters these definitions.

**Scope of the reconstruction.** This is automatic **queen-cascade depth**.
Earlier explicit exclusion deductions are treated as supplied lemmas; their
internal dependencies are not reconstructed. If a placement is not a singleton
in any row/column/region, its advanced explanation, such as lookahead, becomes
an **opaque seed** at depth 1. Its unknown internal depth is reported rather
than replaced with invented edges. This is therefore not the complete depth
of the puzzle's proof, and its correlation need not improve over $M$.

The two scatter panels compare the metrics on the same puzzles. Black points
show mean solvability and 95% puzzle-bootstrap intervals: quintiles for $M$,
exact integer depths for $D_{\max}$. Source colors match Section 8.1.
The following two graphs show the reconstructed supports for the earlier
counterexamples #1666 and #1610 when available. Arrows denote selected
sufficient supports, which need not be the only possible explanation.

This is an exploratory comparison on the existing test results. The cell reuses
DFM attempts and deduction traces; it evaluates no neural network and generates
no new solution attempts.


In [ ]:
import csv
from _shared.queens_deduction_metrics import queen_dependency_depth


def dependency_spearman(x, y):
    def ranks(values):
        _, inverse, counts = np.unique(values, return_inverse=True, return_counts=True)
        return (np.cumsum(counts) - counts + (counts - 1) / 2)[inverse]
    a, b = ranks(x), ranks(y)
    if len(a) < 2 or np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])


if "evaluation_results" in globals():
    depth_evaluation = evaluation_results
else:
    with np.load(ARTIFACTS / "queens7_dfm_evaluation.npz", allow_pickle=False) as saved:
        depth_evaluation = {key: saved[key].copy() for key in saved.files}
if (
    "dataset_sha256" in depth_evaluation
    and str(depth_evaluation["dataset_sha256"]) != puzzles["metadata"]["data_sha256"]
):
    raise ValueError("DFM evaluation and dataset do not match.")
depth_indices = np.asarray(depth_evaluation["dataset_indices"], dtype=np.int64)
depth_attempts = np.asarray(depth_evaluation["successes"], dtype=bool)
if (
    depth_indices.ndim != 1 or not len(depth_indices)
    or len(np.unique(depth_indices)) != len(depth_indices)
    or not np.isin(depth_indices, splits["test"]).all()
    or depth_attempts.ndim != 2 or depth_attempts.shape[0] != len(depth_indices)
    or depth_attempts.shape[1] < 1
):
    raise ValueError("Expected aligned DFM attempts for distinct current test puzzles.")
if "deduction_records" in globals():
    depth_trace_records = deduction_records
else:
    depth_trace_records = [
        json.loads(line) for line in (
            ARTIFACTS / "queens7_dfm_deductive_comparison.jsonl"
        ).read_text(encoding="utf-8").splitlines() if line.strip()
    ]
depth_trace_lookup = {row["fingerprint"]: row for row in depth_trace_records}
depth_records, depth_skipped = [], []
depth_progress = widgets.IntProgress(
    value=0, max=len(depth_indices), description="Dependency graphs",
    style={"description_width": "initial"}, layout=widgets.Layout(width="100%"),
)
display(depth_progress)
for position, index in enumerate(depth_indices):
    index = int(index)
    fingerprint = str(puzzles["fingerprints"][index])
    record = depth_trace_lookup.get(fingerprint)
    if record is None or "deduction_steps" not in record:
        raise ValueError(f"Missing trace for puzzle {index}; rerun Section 8.")
    if not record["is_solved_deductively"]:
        depth_skipped.append(index)
    else:
        graph = queen_dependency_depth(regions[index], record["deduction_steps"])
        depth_records.append(dict(
            dataset_index=index, fingerprint=fingerprint,
            source=str(puzzles["source_names"][index]),
            source_id=str(puzzles["source_ids"][index]),
            diffusion_successes=int(depth_attempts[position].sum()),
            diffusion_attempts=int(depth_attempts.shape[1]),
            diffusion_solvability=float(depth_attempts[position].mean()),
            **graph,
        ))
    depth_progress.value = position + 1
depth_progress.bar_style = "success"
if not depth_records:
    raise ValueError("No complete traces are available.")

depth_rates = np.asarray([row["diffusion_solvability"] for row in depth_records])
depth_sources = np.asarray([row["source"] for row in depth_records])
depth_waits = np.asarray([row["mean_wait"] for row in depth_records])
depth_maxima = np.asarray([row["max_depth"] for row in depth_records])
depth_means = np.asarray([row["mean_depth"] for row in depth_records])
depth_correlations = {
    "mean_wait": dependency_spearman(depth_waits, depth_rates),
    "max_depth": dependency_spearman(depth_maxima, depth_rates),
    "mean_depth": dependency_spearman(depth_means, depth_rates),
}
depth_source_statistics = []
for source in sorted(set(depth_sources.tolist())):
    selected = depth_sources == source
    depth_source_statistics.append(dict(
        source=source, count=int(selected.sum()),
        **{key: dependency_spearman(values[selected], depth_rates[selected])
           for key, values in (
               ("mean_wait", depth_waits), ("max_depth", depth_maxima),
               ("mean_depth", depth_means),
           )},
    ))
depth_palette = {
    "Synthetic": "#5267DB", "Queens.game": "#D88437",
    "AMAZE": "#2F9D88", "Queendom": "#9B6BD3",
}
depth_comparison_figure = make_subplots(
    rows=1, cols=2, horizontal_spacing=0.11,
    subplot_titles=[
        f"Mean pre-queen eliminations · ρ = {depth_correlations['mean_wait']:+.3f}",
        f"Maximum cascade depth · ρ = {depth_correlations['max_depth']:+.3f}",
    ],
)
depth_bootstrap_rng = np.random.default_rng(SEED + 805)
depth_calibration = {}
for col, key, values in (
    (1, "mean_wait", depth_waits), (2, "max_depth", depth_maxima),
):
    for source in sorted(set(depth_sources.tolist())):
        selected = np.flatnonzero(depth_sources == source)
        custom = [[
            str(depth_records[i]["dataset_index"]), depth_records[i]["source_id"],
            str(depth_records[i]["diffusion_successes"]),
            str(depth_records[i]["diffusion_attempts"]),
            str(depth_records[i]["queen_depths"]),
            str(depth_records[i]["opaque_seed_count"]),
        ] for i in selected]
        depth_comparison_figure.add_trace(go.Scatter(
            x=values[selected], y=100 * depth_rates[selected],
            mode="markers", name=source, legendgroup=source, showlegend=col == 1,
            marker=dict(size=8, opacity=0.6, color=depth_palette.get(source, "#64748B")),
            customdata=custom,
            hovertemplate=(
                "Puzzle %{customdata[0]} · %{customdata[1]}<br>Metric: %{x:.3f}"
                "<br>DFM solvability: %{y:.3f}% (%{customdata[2]}/%{customdata[3]})"
                "<br>Queen depths: %{customdata[4]}"
                "<br>Opaque seeds: %{customdata[5]}<extra>%{fullData.name}</extra>"
            ),
        ), row=1, col=col)
    if key == "mean_wait":
        edges = np.unique(np.quantile(values, np.linspace(0, 1, 6)))
        groups = np.searchsorted(edges[1:-1], values, side="right")
    else:
        groups = values
    rows = []
    for group in np.unique(groups):
        selected = groups == group
        probabilities = depth_rates[selected]
        draws = depth_bootstrap_rng.integers(
            0, len(probabilities), size=(2000, len(probabilities))
        )
        low, high = np.quantile(probabilities[draws].mean(axis=1), [0.025, 0.975])
        rows.append(dict(
            metric=float(values[selected].mean()), probability=float(probabilities.mean()),
            lower=float(low), upper=float(high), count=int(selected.sum()),
        ))
    depth_calibration[key] = rows
    depth_comparison_figure.add_trace(go.Scatter(
        x=[r["metric"] for r in rows], y=[100 * r["probability"] for r in rows],
        mode="lines+markers", name="Group mean · 95% interval",
        legendgroup="means", showlegend=col == 1,
        line=dict(color="#172033", width=3), marker=dict(size=9, symbol="diamond"),
        error_y=dict(
            type="data", symmetric=False,
            array=[100 * (r["upper"] - r["probability"]) for r in rows],
            arrayminus=[100 * (r["probability"] - r["lower"]) for r in rows],
        ),
        customdata=[r["count"] for r in rows],
        hovertemplate="Metric %{x:.3f}<br>Mean solvability %{y:.2f}%"
                      "<br>%{customdata} puzzles<extra></extra>",
    ), row=1, col=col)
depth_comparison_figure.update_xaxes(
    title_text="Mean pre-queen eliminations · M", rangemode="tozero", row=1, col=1
)
depth_comparison_figure.update_xaxes(
    title_text="Longest queen dependency chain · Dmax", dtick=1, row=1, col=2
)
depth_comparison_figure.update_yaxes(range=[-2, 102], dtick=20)
depth_comparison_figure.update_yaxes(
    title_text="DFM solvability (%)", row=1, col=1
)
depth_comparison_figure.update_layout(
    title=dict(text="Deduction metrics on the same held-out puzzles", x=0),
    template="plotly_white", autosize=True, height=540,
    legend=dict(orientation="h", x=0, y=-0.23),
    margin=dict(l=65, r=20, t=100, b=125),
    font=dict(family="Arial, sans-serif", size=15),
)
depth_comparison_figure.show(config=PLOT_CONFIG)

def dependency_graph_positions(graph):
    positions = {}
    for level in sorted(set(graph["queen_depths"])):
        nodes = [q["queen_id"] for q in graph["queens"] if q["depth"] == level]
        offsets = (np.arange(len(nodes)) - (len(nodes) - 1) / 2).tolist()
        positions.update({node: (level, offset) for node, offset in zip(nodes, offsets)})
    return positions


depth_lookup = {row["dataset_index"]: row for row in depth_records}
depth_examples = [index for index in (1666, 1610) if index in depth_lookup]
if not depth_examples:
    depth_examples = [row["dataset_index"] for row in depth_records[:2]]
depth_graph_figure = make_subplots(
    rows=1, cols=len(depth_examples), horizontal_spacing=0.13,
    subplot_titles=[
        f"Puzzle {index} · {depth_lookup[index]['diffusion_successes']}/"
        f"{depth_lookup[index]['diffusion_attempts']} solved<br>"
        f"M={depth_lookup[index]['mean_wait']:.2f} · "
        f"Dmax={depth_lookup[index]['max_depth']} · "
        f"Dmean={depth_lookup[index]['mean_depth']:.2f}"
        for index in depth_examples
    ],
)
for col, index in enumerate(depth_examples, start=1):
    graph = depth_lookup[index]
    positions = dependency_graph_positions(graph)
    subplot = depth_graph_figure.get_subplot(1, col)
    xref = subplot.xaxis.plotly_name.replace("axis", "")
    yref = subplot.yaxis.plotly_name.replace("axis", "")
    for parent, child in graph["edges"]:
        x0, y0 = positions[parent]
        x1, y1 = positions[child]
        depth_graph_figure.add_annotation(
            x=x1, y=y1, ax=x0, ay=y0, xref=xref, yref=yref, axref=xref, ayref=yref,
            text="", showarrow=True, arrowhead=2, arrowsize=1,
            arrowwidth=1.3, arrowcolor="#94A3B8", standoff=17, startstandoff=17,
        )
    queens = graph["queens"]
    depth_graph_figure.add_trace(go.Scatter(
        x=[positions[q["queen_id"]][0] for q in queens],
        y=[positions[q["queen_id"]][1] for q in queens],
        mode="markers+text", text=[str(q["queen_id"] + 1) for q in queens],
        textfont=dict(size=13, color="#243355"),
        marker=dict(
            size=34, color="#E8EEFD", line=dict(color="#5267DB", width=1.5),
            symbol=["diamond" if q["opaque_seed"] else "circle" for q in queens],
        ),
        customdata=[[q["row"] + 1, q["column"] + 1, q["depth"], q["reason"]]
                    for q in queens],
        hovertemplate="Queen at row %{customdata[0]}, column %{customdata[1]}"
                      "<br>Depth %{customdata[2]}<br>%{customdata[3]}<extra></extra>",
        showlegend=False,
    ), row=1, col=col)
    depth_graph_figure.update_xaxes(
        title_text="Cascade depth", dtick=1, range=[0.5, graph["max_depth"] + 0.5],
        showgrid=False, zeroline=False, row=1, col=col,
    )
    depth_graph_figure.update_yaxes(
        range=[-2, 2], visible=False, row=1, col=col,
    )
depth_graph_figure.update_layout(
    title=dict(text="Selected queen dependencies · numbers denote placement order", x=0),
    template="plotly_white", autosize=True, height=400,
    margin=dict(l=25, r=25, t=125, b=60),
    font=dict(family="Arial, sans-serif", size=15),
)
depth_graph_figure.show(config=PLOT_CONFIG)
depth_opaque_count = sum(row["opaque_seed_count"] for row in depth_records)
display(HTML(
    f"<b>Spearman:</b> M = {depth_correlations['mean_wait']:+.3f} · "
    f"Dmax = {depth_correlations['max_depth']:+.3f} · "
    f"Dmean = {depth_correlations['mean_depth']:+.3f}"
    f"<br>Opaque advanced-rule placements: {depth_opaque_count}/"
    f"{len(depth_records) * BOARD_SIZE}. Incomplete traces excluded: {len(depth_skipped)}."
    "<br>Cascade depth should be judged by this comparison; a longer dependency "
    "chain is not assumed to predict lower DFM solvability."
))

def depth_json_default(value):
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(f"Unsupported result type: {type(value).__name__}")


depth_artifact = dict(
    definition="queen_cascade_dependency_depth_v1",
    dataset_sha256=puzzles["metadata"]["data_sha256"],
    diffusion_steps=int(depth_evaluation["steps"]),
    correlations={key: None if not np.isfinite(value) else value
                  for key, value in depth_correlations.items()},
    source_statistics=[
        {key: None if isinstance(value, float) and not np.isfinite(value) else value
         for key, value in row.items()}
        for row in depth_source_statistics
    ],
    calibration=depth_calibration,
    opaque_seed_count=depth_opaque_count,
    excluded_incomplete_indices=depth_skipped,
    puzzles=depth_records,
)
(ARTIFACTS / "queens7_dependency_depth.json").write_text(
    json.dumps(depth_artifact, ensure_ascii=False, indent=2, default=depth_json_default)
    + "\n", encoding="utf-8",
)
depth_csv_fields = [
    "dataset_index", "source", "source_id", "diffusion_successes", "diffusion_attempts",
    "diffusion_solvability", "mean_wait", "max_depth", "mean_depth", "queen_depths",
    "opaque_seed_count", "root_count",
]
with (ARTIFACTS / "queens7_dependency_depth.csv").open(
    "w", encoding="utf-8-sig", newline="",
) as output:
    writer = csv.DictWriter(output, fieldnames=depth_csv_fields, extrasaction="ignore")
    writer.writeheader()
    writer.writerows(depth_records)
depth_comparison_figure.write_html(
    ARTIFACTS / "queens7_dependency_depth.html", include_plotlyjs=True, config=PLOT_CONFIG,
)
depth_graph_figure.write_html(
    ARTIFACTS / "queens7_dependency_graphs.html", include_plotlyjs=True, config=PLOT_CONFIG,
)


### 8.3 Deduction feature vector and mutual information

Represent each completed trace by 51 numeric features:

- Counts of elimination batches, local contradictions, subset rules,
  multi-zone subsets, X-Wing and lookahead.
- Rule types placing queens, rule diversity and transitions.
- Waiting counts before individual queens and early/late queen placements;
  lengths of consecutive elimination/placement runs.
- Maximum/mean cascade depth, roots, edges, layer width and opaque seeds.
- Spatial spans of the cells named in elimination batches.

The vector contains no DFM outputs, dataset source, puzzle ID, absolute color
IDs or supplied success probability. Cascade features also use the original
region map, as in Section 8.2. Only completed traces are included.

The measured target is $\widehat p_i=c_i/A$, not the unknown true success
probability. Here $A=32$, so measurements have finite-attempt noise.

#### Information of individual features

With 256 puzzles, direct frequency counting of whole 51-dimensional vectors
would mostly memorize individual observations. For a stable descriptive
comparison, discretize measured solvability into four fixed intervals:

$$
B(\widehat p)=
\begin{cases}
0,&0\le\widehat p<0.25,\\
1,&0.25\le\widehat p<0.50,\\
2,&0.50\le\widehat p<0.75,\\
3,&0.75\le\widehat p\le1.
\end{cases}
$$

Discretize each feature $X_j$ into at most four quantile bins, preserving
distinct values when it has at most four. Estimate the discrete information:

$$
\widehat I(\widetilde X_j;B)
=\sum_{a,b}\widehat p(a,b)
 \log_2\frac{\widehat p(a,b)}{\widehat p(a)\widehat p(b)}.
$$

The plot subtracts the mean MI from 499 shuffled-target runs to expose the
finite-sample baseline. Hover shows the raw MI and shuffled mean.
This subtraction is a descriptive bias diagnostic, not an exact unbiased
estimator or a significance test accounting for related puzzles.
Identical discretized features are grouped in the plot. **Do not sum these
MI values:** waiting features and rule counts overlap.

#### Information carried by the whole vector

The following section trains a separate four-class probability model
$q_\theta(B\mid X)$ on other queen-placement groups. Its out-of-fold
cross-entropy gives a model-dependent variational lower-bound estimate:

$$
I(X;\widehat p)\ \ge\ I(X;B)
\ \ge\ H(B)+\mathbb E[\log_2 q_\theta(B\mid X)].
$$

Use the empirical entropy and held-out log probabilities to estimate the
right-hand side. It can be negative when prediction/calibration is poor;
that does not mean mutual information is negative. This quantifies information
the chosen model can extract, rather than claiming an exact high-dimensional
MI estimate. Binning also leaves within-bin information unmeasured.
The fitted lower bound can be below a one-feature plug-in estimate; it is
not the total MI and the fitted classifier need not attain the optimal bound.


In [ ]:
import csv
import hashlib
import joblib
import sklearn
from _shared.queens_solvability_prediction import (
    build_feature_dataset, feature_information, fit_solvability_models,
    source_information_audit,
)

META_CV_SEED = SEED + 806
MI_PERMUTATIONS = 499
if "evaluation_results" in globals():
    predictor_evaluation = evaluation_results
else:
    with np.load(ARTIFACTS / "queens7_dfm_evaluation.npz", allow_pickle=False) as saved:
        predictor_evaluation = {key: saved[key].copy() for key in saved.files}
if "deduction_records" in globals():
    predictor_traces = deduction_records
else:
    predictor_traces = [
        json.loads(line) for line in (
            ARTIFACTS / "queens7_dfm_deductive_comparison.jsonl"
        ).read_text(encoding="utf-8").splitlines() if line.strip()
    ]
predictor_data = build_feature_dataset(puzzles, predictor_evaluation, predictor_traces)
if not np.isin(predictor_data["dataset_indices"], splits["test"]).all():
    raise ValueError("Expected the current DFM test puzzles.")
predictor_information = feature_information(
    predictor_data["X"], predictor_data["y"], predictor_data["feature_names"],
    permutations=MI_PERMUTATIONS, seed=META_CV_SEED,
)
predictor_hash = hashlib.sha256()
for value in (
    predictor_data["X"], predictor_data["y"], predictor_data["groups"],
    predictor_data["dataset_indices"],
):
    predictor_hash.update(np.ascontiguousarray(value).tobytes())
predictor_hash.update(json.dumps(predictor_data["feature_names"]).encode())
predictor_hash.update(str(int(predictor_evaluation["steps"])).encode())
predictor_signature = predictor_hash.hexdigest()

# Group equivalent discretizations for display; retain all 51 original features.
mi_unique_rows, mi_seen_codes = [], {}
for row in predictor_information["features"]:
    column = predictor_data["feature_names"].index(row["feature"])
    values = predictor_data["X"][:, column]
    unique = np.unique(values)
    if len(unique) <= 4:
        codes = np.searchsorted(unique, values)
    else:
        edges = np.unique(np.quantile(values, np.linspace(0, 1, 5)))
        codes = np.searchsorted(edges[1:-1], values, side="right")
    signature = tuple(codes.tolist())
    if signature in mi_seen_codes:
        mi_seen_codes[signature]["equivalent_features"].append(row["feature"])
    else:
        shown = dict(row, equivalent_features=[row["feature"]])
        mi_seen_codes[signature] = shown
        mi_unique_rows.append(shown)
mi_shown = mi_unique_rows[:12]
predictor_mi_figure = go.Figure(go.Bar(
    x=[row["excess_bits"] for row in mi_shown],
    y=[row["feature"].replace("_", " ") for row in mi_shown],
    orientation="h", marker_color="#5267DB",
    customdata=[[
        row["mi_bits"], row["null_mean_bits"], row["null_95_bits"],
        ", ".join(row["equivalent_features"]),
    ] for row in mi_shown],
    hovertemplate=(
        "%{y}<br>MI above shuffled mean: %{x:.4f} bits"
        "<br>Raw MI: %{customdata[0]:.4f} bits"
        "<br>Shuffled mean: %{customdata[1]:.4f} bits"
        "<br>Shuffled 95th percentile: %{customdata[2]:.4f} bits"
        "<br>Equivalent binned features: %{customdata[3]}<extra></extra>"
    ),
))
predictor_mi_figure.update_layout(
    title=dict(text="Individual deduction features · information about solvability bins", x=0),
    template="plotly_white", autosize=True, height=550,
    xaxis_title="MI minus mean shuffled-target MI (bits)",
    yaxis=dict(autorange="reversed"),
    margin=dict(l=240, r=25, t=85, b=65),
    font=dict(family="Arial, sans-serif", size=15),
)
predictor_mi_figure.show(config=PLOT_CONFIG)
predictor_bin_counts = np.bincount(predictor_information["target_bins"], minlength=4)
display(HTML(
    f"<b>{len(predictor_data['y'])} puzzles · "
    f"{predictor_data['X'].shape[1]} features · "
    f"{len(np.unique(predictor_data['groups']))} queen-placement symmetry groups.</b>"
    f"<br>Solvability-bin counts: {predictor_bin_counts.tolist()} · "
    f"H(B) = {predictor_information['entropy_bits']:.3f} bits."
    "<br>Source labels are retained only for auditing and plot colors."
))
with (ARTIFACTS / "queens7_deduction_features.csv").open(
    "w", encoding="utf-8-sig", newline="",
) as output:
    fields = ["dataset_index", "source", "solution_group", "observed_solvability"]
    writer = csv.DictWriter(output, fieldnames=fields + predictor_data["feature_names"])
    writer.writeheader()
    for i in range(len(predictor_data["y"])):
        writer.writerow(dict(
            dataset_index=int(predictor_data["dataset_indices"][i]),
            source=predictor_data["source_names"][i],
            solution_group=predictor_data["groups"][i],
            observed_solvability=float(predictor_data["y"][i]),
            **dict(zip(predictor_data["feature_names"], predictor_data["X"][i])),
        ))
with (ARTIFACTS / "queens7_feature_information.csv").open(
    "w", encoding="utf-8-sig", newline="",
) as output:
    writer = csv.DictWriter(output, fieldnames=list(predictor_information["features"][0]))
    writer.writeheader()
    writer.writerows(predictor_information["features"])
predictor_mi_figure.write_html(
    ARTIFACTS / "queens7_feature_information.html",
    include_plotlyjs=True, config=PLOT_CONFIG,
)


### 8.4 Predicting DFM solvability: tree, forest and gradient boosting

Fit regressors to the measured probability $\widehat p_i$:

- A decision tree, tuned over depths 2/3/5 and leaf sizes 8/16.
- A 200-tree random forest, tuned over depths 5/unlimited and leaf sizes 4/8/16.
- Histogram gradient boosting, with 150 iterations, learning rate 0.05,
  leaf sizes at least 12, 3/7 leaves and L2 regularization 1/10.

A training-mean predictor is the baseline. No MI-based feature selection is
performed. MI plots describe the same data, but their target-dependent rankings
are not fed into model fitting.

#### Grouped nested cross-validation

Group puzzles that share their queen placement up to rotations/reflections.
This keeps repeated solutions out of both sides of an evaluation split.
In the current collection, 256 measured puzzles form 79 such groups.

Use five outer group folds. Tune hyperparameters using three inner group
folds on the outer training data only. Every plotted prediction comes from
a model that has seen neither that puzzle nor its queen-placement group.
Compare RMSE and MAE in **percentage points**, $R^2$, and Spearman correlation.

A separate random-forest classifier estimates the four solvability-bin
probabilities for the joint-information diagnostic in Section 8.3.
It follows the same outer groups, with inner tuning by log loss.
Bootstrap the 79 solution groups for intervals, keeping related puzzles
together. These intervals condition on the fitted out-of-fold predictions;
they do not include all uncertainty from retraining or the choice of split.

After evaluation, refit all three regressors and the information classifier
on all measured puzzles for future use. The cached bundle contains their
feature names and models. The expandable tree describes this final refit;
the scatter and error bars use outer-fold predictions.

This remains exploratory analysis of the current DFM test results. The target
fractions are noisy measurements from 32 attempts, and the feature design was
informed by earlier inspection of this collection.

As a confounding control, also predict each held-out puzzle by the mean of
its source among the outer training puzzles. Source labels are used only in
this audit baseline. Report $I(\mathrm{source};B)$ and the analogous bound
$H(B\mid\mathrm{source})+\mathbb E[\log_2q_\theta(B\mid X)]$.
A near-zero or negative value means this classifier has not demonstrated
additional information beyond source; it does not prove conditional MI is zero.

Completed results are reused when the feature matrix, targets, puzzle groups,
scikit-learn version, and CV settings match. Set
<code>PREDICTOR_REFIT = True</code> to rerun the fitting.


In [ ]:
META_OUTER_SPLITS = 5
META_INNER_SPLITS = 3
META_BOOTSTRAP_DRAWS = 2000
PREDICTOR_REFIT = False
PREDICTOR_PATH = ARTIFACTS / "queens7_solvability_predictors.joblib"
predictor_bundle = None
if PREDICTOR_PATH.exists() and not PREDICTOR_REFIT:
    candidate = joblib.load(PREDICTOR_PATH)
    if (
        candidate.get("schema") == "queens_trace_predictor_v1"
        and candidate.get("signature") == predictor_signature
        and candidate.get("sklearn_version") == sklearn.__version__
        and candidate.get("seed") == META_CV_SEED
        and candidate.get("outer_splits") == META_OUTER_SPLITS
        and candidate.get("inner_splits") == META_INNER_SPLITS
        and candidate.get("bootstrap_draws") == META_BOOTSTRAP_DRAWS
    ):
        predictor_bundle = candidate
if predictor_bundle is None:
    predictor_progress = widgets.IntProgress(
        value=0, max=1, description="Grouped model fitting",
        style={"description_width": "initial"}, layout=widgets.Layout(width="100%"),
    )
    predictor_status = widgets.HTML()
    display(predictor_progress, predictor_status)

    def update_predictor(stage, done, total):
        predictor_progress.max = total
        predictor_progress.value = done
        predictor_status.value = html.escape(stage)

    predictor_fitted = fit_solvability_models(
        predictor_data, seed=META_CV_SEED,
        outer_splits=META_OUTER_SPLITS, inner_splits=META_INNER_SPLITS,
        bootstrap_draws=META_BOOTSTRAP_DRAWS, progress=update_predictor,
    )
    predictor_progress.bar_style = "success"
    predictor_bundle = dict(
        schema="queens_trace_predictor_v1", signature=predictor_signature,
        sklearn_version=sklearn.__version__, data=predictor_data,
        information=predictor_information, fitted=predictor_fitted,
        seed=META_CV_SEED, outer_splits=META_OUTER_SPLITS,
        inner_splits=META_INNER_SPLITS, bootstrap_draws=META_BOOTSTRAP_DRAWS,
    )
    joblib.dump(predictor_bundle, PREDICTOR_PATH, compress=3)
else:
    predictor_fitted = predictor_bundle["fitted"]
    display(HTML("<b>Loaded completed grouped cross-validation and fitted predictors.</b>"))

predictor_labels = {
    "mean": "Training mean", "tree": "Decision tree",
    "forest": "Random forest", "boosting": "Gradient boosting",
    "source_mean": "Source mean · audit",
}
predictor_metrics = {row["model"]: row for row in predictor_fitted["metrics"]}
predictor_source_audit = source_information_audit(
    predictor_data, predictor_fitted,
    bootstrap_draws=META_BOOTSTRAP_DRAWS, seed=META_CV_SEED + 2,
)
predictor_metrics["source_mean"] = predictor_source_audit["source_baseline"]
predictor_output_predictions = dict(
    predictor_fitted["predictions"],
    source_mean=predictor_source_audit["source_mean_predictions"],
)
predictor_palette = {
    "Synthetic": "#5267DB", "Queens.game": "#D88437",
    "AMAZE": "#2F9D88", "Queendom": "#9B6BD3",
}
predictor_scatter = make_subplots(
    rows=1, cols=3, horizontal_spacing=0.06,
    subplot_titles=[
        f"{predictor_labels[name]} · OOF R²={predictor_metrics[name]['r2']:+.3f}"
        for name in ("tree", "forest", "boosting")
    ],
)
for col, name in enumerate(("tree", "forest", "boosting"), start=1):
    predictor_scatter.add_trace(go.Scatter(
        x=[0, 100], y=[0, 100], mode="lines", line=dict(color="#CBD5E1", dash="dash"),
        showlegend=False, hoverinfo="skip",
    ), row=1, col=col)
    for source in sorted(set(predictor_data["source_names"].tolist())):
        selected = np.flatnonzero(predictor_data["source_names"] == source)
        custom = [[
            str(predictor_data["dataset_indices"][i]),
            str(predictor_fitted["fold_id"][i] + 1),
        ] for i in selected]
        predictor_scatter.add_trace(go.Scatter(
            x=100 * predictor_data["y"][selected],
            y=100 * predictor_fitted["predictions"][name][selected],
            mode="markers", name=source, legendgroup=source, showlegend=col == 1,
            marker=dict(size=7, opacity=0.65, color=predictor_palette.get(source, "#64748B")),
            customdata=custom,
            hovertemplate="Puzzle %{customdata[0]} · outer fold %{customdata[1]}"
                          "<br>Observed %{x:.2f}%<br>Predicted %{y:.2f}%"
                          "<extra>%{fullData.name}</extra>",
        ), row=1, col=col)
predictor_scatter.update_xaxes(title_text="Observed DFM solvability (%)",
                               range=[-2, 102], dtick=25)
predictor_scatter.update_yaxes(range=[-2, 102], dtick=25)
predictor_scatter.update_yaxes(title_text="Predicted solvability (%)", row=1, col=1)
predictor_scatter.update_layout(
    title=dict(text="Prediction from deduction traces · nested grouped out-of-fold results", x=0),
    template="plotly_white", autosize=True, height=500,
    legend=dict(orientation="h", x=0, y=-0.25),
    margin=dict(l=70, r=20, t=105, b=125),
    font=dict(family="Arial, sans-serif", size=14),
)
predictor_scatter.show(config=PLOT_CONFIG)

predictor_order = ["mean", "source_mean", "tree", "forest", "boosting"]
predictor_rmse_figure = go.Figure(go.Bar(
    x=[predictor_labels[name] for name in predictor_order],
    y=[predictor_metrics[name]["rmse_pp"] for name in predictor_order],
    marker_color=["#A0A8B5", "#64748B", "#D88437", "#5267DB", "#2F9D88"],
    error_y=dict(
        type="data", symmetric=False,
        array=[predictor_metrics[name]["rmse_upper_pp"] - predictor_metrics[name]["rmse_pp"]
               for name in predictor_order],
        arrayminus=[predictor_metrics[name]["rmse_pp"] - predictor_metrics[name]["rmse_lower_pp"]
                    for name in predictor_order],
    ),
    hovertemplate="%{x}<br>OOF RMSE: %{y:.2f} percentage points<extra></extra>",
))
predictor_rmse_figure.update_layout(
    title="OOF error · 95% solution-group bootstrap intervals",
    template="plotly_white", autosize=True, height=370,
    yaxis_title="RMSE (percentage points) · lower is better",
    margin=dict(l=75, r=20, t=75, b=60),
    font=dict(family="Arial, sans-serif", size=15),
)
predictor_rmse_figure.show(config=PLOT_CONFIG)
predictor_summary = []
for name in predictor_order:
    row = predictor_metrics[name]
    predictor_summary.append(
        f"<b>{predictor_labels[name]}</b>: R²={row['r2']:+.3f} · "
        f"RMSE={row['rmse_pp']:.2f} pp · MAE={row['mae_pp']:.2f} pp · "
        f"Spearman={row['spearman']:+.3f}"
        if row["spearman"] is not None else
        f"<b>{predictor_labels[name]}</b>: R²={row['r2']:+.3f} · "
        f"RMSE={row['rmse_pp']:.2f} pp · MAE={row['mae_pp']:.2f} pp"
    )
predictor_joint = predictor_fitted["information"]
predictor_summary.append(
    f"<br><b>Whole-vector information:</b> H(B)={predictor_joint['entropy_bits']:.3f} bits · "
    f"OOF cross-entropy={predictor_joint['oof_cross_entropy_bits']:.3f} bits."
    f"<br>Variational lower-bound estimate: "
    f"<b>{predictor_joint['joint_lower_bound_bits']:.3f} bits</b> · "
    f"95% group-bootstrap interval "
    f"[{predictor_joint['bootstrap_interval_bits'][0]:.3f}, "
    f"{predictor_joint['bootstrap_interval_bits'][1]:.3f}]."
)
predictor_best_name = min(("tree", "forest", "boosting"),
                          key=lambda name: predictor_metrics[name]["mse"])
predictor_summary.append(
    f"<br><b>Source information:</b> "
    f"I(source;B)={predictor_source_audit['source_information_bits']:.3f} bits."
    f"<br>After conditioning on source, the classifier's lower-bound estimate is "
    f"<b>{predictor_source_audit['conditional_lower_bound_bits']:.3f} bits</b> · "
    f"95% interval [{predictor_source_audit['conditional_lower_bound_interval'][0]:.3f}, "
    f"{predictor_source_audit['conditional_lower_bound_interval'][1]:.3f}]."
)
predictor_summary.append("<br><b>Source audit for the lowest-OOF-RMSE regressor:</b>")
for source in sorted(set(predictor_data["source_names"].tolist())):
    selected = predictor_data["source_names"] == source
    actual = predictor_data["y"][selected]
    predicted = predictor_fitted["predictions"][predictor_best_name][selected]
    mse = float(np.mean((actual - predicted)**2))
    r2 = 1 - mse / np.var(actual) if len(actual) > 1 and np.var(actual) else None
    predictor_summary.append(
        f"{html.escape(source)}: n={len(actual)} · RMSE={100 * np.sqrt(mse):.2f} pp"
        + (f" · R²={r2:+.3f}" if r2 is not None else "")
    )
display(HTML("<br>".join(predictor_summary)))
display(HTML(
    "<details><summary>Decision tree refitted on all measured puzzles</summary>"
    "<pre style='max-height:400px;overflow:auto'>"
    + html.escape(predictor_fitted["tree_text"]) + "</pre></details>"
))

def predictor_json_default(value):
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    raise TypeError(type(value).__name__)


predictor_export = dict(
    schema="queens_trace_predictor_v1", signature=predictor_signature,
    feature_names=predictor_data["feature_names"],
    sample_count=len(predictor_data["y"]), group_count=predictor_fitted["group_count"],
    feature_information=predictor_information,
    model_metrics=[predictor_metrics[name] for name in predictor_order],
    information=predictor_joint, source_audit=predictor_source_audit,
    final_parameters=predictor_fitted["final_parameters"],
    fold_parameters=predictor_fitted["fold_parameters"],
    dataset_sha256=puzzles["metadata"]["data_sha256"],
)
(ARTIFACTS / "queens7_solvability_prediction.json").write_text(
    json.dumps(predictor_export, ensure_ascii=False, indent=2, default=predictor_json_default)
    + "\n", encoding="utf-8",
)
with (ARTIFACTS / "queens7_solvability_predictions.csv").open(
    "w", encoding="utf-8-sig", newline="",
) as output:
    fields = ["dataset_index", "source", "solution_group", "outer_fold",
              "observed_solvability", *predictor_order]
    writer = csv.DictWriter(output, fieldnames=fields)
    writer.writeheader()
    for i in range(len(predictor_data["y"])):
        writer.writerow(dict(
            dataset_index=int(predictor_data["dataset_indices"][i]),
            source=predictor_data["source_names"][i],
            solution_group=predictor_data["groups"][i],
            outer_fold=int(predictor_fitted["fold_id"][i]),
            observed_solvability=float(predictor_data["y"][i]),
            **{name: float(predictor_output_predictions[name][i]) for name in predictor_order},
        ))
(ARTIFACTS / "queens7_solvability_tree.txt").write_text(
    predictor_fitted["tree_text"], encoding="utf-8",
)
predictor_scatter.write_html(
    ARTIFACTS / "queens7_solvability_predictions.html",
    include_plotlyjs=True, config=PLOT_CONFIG,
)
predictor_rmse_figure.write_html(
    ARTIFACTS / "queens7_solvability_model_errors.html",
    include_plotlyjs=True, config=PLOT_CONFIG,
)
